## Block 1: Imports and Global Config

Imports core dependencies and defines all configurable experiment variables in one place.


In [ ]:
import os, sys, json, csv, math, time, subprocess, warnings, re
from pathlib import Path

import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix

import warnings
warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True",
    module="torch.nn.modules.transformer"
)

# Fallback for non-notebook execution contexts.
try:
    display  # type: ignore[name-defined]
except NameError:
    def display(value):
        print(value)

# ---------------- Global Config (edit here) ----------------
# Runtime / environment
def _parse_env_int(env_name, default_value):
    raw_value = os.environ.get(env_name, "").strip()
    if not raw_value:
        return int(default_value)
    try:
        return int(raw_value)
    except Exception:
        print(f"Warning: invalid integer for {env_name}='{raw_value}', using default={default_value}")
        return int(default_value)


def _parse_env_int_list(env_name, default_values):
    raw_value = os.environ.get(env_name, "").strip()
    if not raw_value:
        return list(default_values)
    try:
        parsed_values = [int(part.strip()) for part in raw_value.split(",") if part.strip()]
        return parsed_values if parsed_values else list(default_values)
    except Exception:
        print(f"Warning: invalid int-list for {env_name}='{raw_value}', using default={default_values}")
        return list(default_values)


PREFERRED_GPU_INDEX = _parse_env_int("PREFERRED_GPU_INDEX", 1)
# If non-empty, use exactly these CUDA device indices (e.g., [0, 1]).
CUDA_DEVICE_INDICES = _parse_env_int_list("CUDA_DEVICE_INDICES", [0, 1, 2])
# Set to "auto", "cuda", "mps", or "cpu".
DEVICE_OVERRIDE = os.environ.get("DEVICE_OVERRIDE", "auto")
AUTO_INSTALL_PACKAGES = ("optuna", "tqdm", "matplotlib", "scikit-learn", "pandas", "numpy", "torch", "seaborn", "aequitas", "xformers", "ipywidgets")

# Experiment identity and paths
EXPERIMENT_TAG = os.environ.get("EXPERIMENT_TAG", "VII").strip() or "VII"
MODEL_HEAD_TYPE = "dense"
RESULTS_DIR = Path(".")
DATASET_DIR = Path("baf-datasets")
DATASET_FILE_NAME = f"{EXPERIMENT_TAG}.csv"
DATA_CSV_PATH = DATASET_DIR / DATASET_FILE_NAME

# Organized output layout (easy to navigate per run)
RUN_OUTPUT_ROOT = RESULTS_DIR / f"ft_dense_baseline_100_run_{EXPERIMENT_TAG}"
OPTUNA_DIR = RUN_OUTPUT_ROOT / "optuna"
CSV_EXPORT_DIR = RUN_OUTPUT_ROOT / "csv_exports"
MODEL_EXPORT_DIR = RUN_OUTPUT_ROOT / "model_artifacts"
PAPER_EXPORT_DIR = RUN_OUTPUT_ROOT / "paper_artifacts"
EXPORT_DIR = MODEL_EXPORT_DIR  # Backward-compatible alias used by downstream blocks

# Dataset schema / split controls
TARGET_CANDIDATE_COLUMNS = ["fraud_bool", "fraud", "is_fraud", "label", "target"]
MONTH_COLUMN = "month"
TRAIN_MONTH_MAX = 5
TEST_MONTH_MIN = 6
VALID_SPLIT_RATIO = 0.2
DROP_FEATURE_NAME_FRAGMENTS = {"customer", "id", "uuid"}
INT_AS_CATEGORICAL_MAX_UNIQUE = 50
AGE_GROUP_COLUMNS = ["age", "Age", "AGE", "age_group", "customer_age"]
AGE_THRESHOLD = 50
INCOME_GROUP_COLUMNS = ["income", "Income", "annual_income", "AMT_INCOME_TOTAL"]
EMPLOYMENT_STATUS_GROUP_COLUMNS = ["employment_status", "EmploymentStatus", "NAME_INCOME_TYPE"]
INCOME_GROUP_QUANTILES = [0.0, 0.25, 0.5, 0.75, 1.0]
AEQUITAS_FAIRNESS_ATTRIBUTES = ["age_group", "income_group", "employment_status_group"]

# Training / optimization controls
RANDOM_SEED = 42
WEIGHT_DECAY = 1e-5
FPR_CAP = 0.05
EPSILON = 1e-12

# Single-stage FT-Transformer tuning schedule
TUNING_TRIALS = 80
TUNING_EPOCHS = 8
FINAL_EPOCHS = 20
# Stop a trial early if validation recall (out of how many bad cases we actually caught) does not improve for this many consecutive epochs (using the EPSILON margin).
# This reduces wasted training on plateaued trials while keeping the best checkpoint seen so far.
EARLY_STOP_PATIENCE = 3
#Optuna will not prune (stop early) the first 8 trials. This allows it to gather some initial data on the search space before it starts making pruning decisions.
PRUNER_STARTUP_TRIALS = 8
#Optuna will not prune (stop early) any trial before it has completed 2 epochs. This allows each trial to have a fair chance to show some initial progress before being considered for pruning.
PRUNER_WARMUP_STEPS = 2

OPTUNA_DIRECTION = "maximize"
OPTUNA_STUDY_NAME = f"ft_dense_baseline_recall_at_fpr_cap_{EXPERIMENT_TAG}"
OPTUNA_STORAGE_PATH = OPTUNA_DIR / f"optuna_{OPTUNA_STUDY_NAME}.db"
OPTUNA_STORAGE_URL = f"sqlite:///{OPTUNA_STORAGE_PATH.resolve()}"
OPTUNA_LOAD_IF_EXISTS = True
VALID_TEST_BATCH_MULTIPLIER = 2
DEFAULT_THRESHOLD = 0.5
# Set True to populate per-trial test metrics in the fairness CSV.
# Set False if you want strict no-test-peeking during tuning.
LOG_TEST_DURING_TUNING = True
# Keep existing trial/best/fairness CSV logs when resuming an existing Optuna study.
# Set True only when you explicitly want a fresh local log export.
RESET_TRIAL_LOG_CSVS = False

# Hyperparameter search space (Transformer + optimizer)
HP_D_TOKEN_CHOICES = [16, 32, 48, 64]
HP_N_BLOCKS_MIN = 2
HP_N_BLOCKS_MAX = 6
HP_N_HEADS_CHOICES = [4, 8]
HP_FFN_HIDDEN_CHOICES = [64, 128, 192, 256]
HP_DROPOUT_MIN = 0.0
HP_DROPOUT_MAX = 0.4
HP_LR_MIN = 3e-4
HP_LR_MAX = 3e-3
HP_BATCH_CHOICES = [1024, 2048, 4096, 8192]

# Model defaults
FT_DEFAULT_D_TOKEN = 32
FT_DEFAULT_N_BLOCKS = 4
FT_DEFAULT_N_HEADS = 8
FT_DEFAULT_FFN_HIDDEN = 128
FT_DEFAULT_DROPOUT = 0.2

# Logs and exports
TRIALS_CSV = CSV_EXPORT_DIR / f"ft_dense_baseline_100_trials_{EXPERIMENT_TAG}.csv"
BEST_CSV = CSV_EXPORT_DIR / f"ft_dense_baseline_100_trials_best_{EXPERIMENT_TAG}.csv"
FAIR_CSV = CSV_EXPORT_DIR / f"ft_dense_baseline_100_trials_fairness_{EXPERIMENT_TAG}.csv"
TRIAL_LOG_FIELDS = [
    "stage", "trial", "epoch",
    "d_token", "n_blocks", "n_heads", "ffn_hidden", "dropout",
    "lr", "batch",
    "val_auc", "val_prauc", "val_recall_at_fpr", "val_fpr", "val_threshold"
]
FAIR_LOG_FIELDS = [
    "stage", "trial", "perf_recall_test", "fairness_ratio_test", "thr_star", "fpr_test",
    "recall_val", "fpr_val", "fairness_ratio_val",
    "aeq_fpr_parity_age_test", "aeq_fpr_parity_income_test", "aeq_fpr_parity_employment_status_test",
    "aeq_fpr_parity_age_val", "aeq_fpr_parity_income_val", "aeq_fpr_parity_employment_status_val",
    "attribute_n_distinct_groups", "attribute_is_degenerate",
    "aeq_attribute_n_distinct_groups_age_test", "aeq_attribute_n_distinct_groups_income_test", "aeq_attribute_n_distinct_groups_employment_status_test",
    "aeq_attribute_n_distinct_groups_age_val", "aeq_attribute_n_distinct_groups_income_val", "aeq_attribute_n_distinct_groups_employment_status_val",
    "aeq_attribute_is_degenerate_age_test", "aeq_attribute_is_degenerate_income_test", "aeq_attribute_is_degenerate_employment_status_test",
    "aeq_attribute_is_degenerate_age_val", "aeq_attribute_is_degenerate_income_val", "aeq_attribute_is_degenerate_employment_status_val"
]

# Plot controls
PLOT_FIGSIZE = (7.0, 5.0)
PLOT_DPI = 150
TOP_K_TRIALS = 5
ZOOM_X_QUANTILE = 0.6
ZOOM_Y_LOW_QUANTILE = 0.05
ZOOM_Y_HIGH_QUANTILE = 0.95




## Block 2: Device Selection

Uses `DEVICE_OVERRIDE` (`"auto"`, `"cuda"`, `"mps"`, `"cpu"`) to select runtime hardware.
In `"auto"` mode, priority is CUDA -> MPS -> CPU.


In [ ]:
# ---------------- Repro/Device ----------------
# Device selection policy:
# - DEVICE_OVERRIDE = "auto": CUDA -> MPS -> CPU
# - DEVICE_OVERRIDE = "cuda"/"mps"/"cpu": force a backend if available (fallback with warning)
# - If CUDA_DEVICE_INDICES is non-empty, use exactly those GPUs (first becomes active device)

device_override_normalized = str(DEVICE_OVERRIDE).strip().lower()
if device_override_normalized not in {"auto", "cuda", "mps", "cpu"}:
    print(f'Warning: DEVICE_OVERRIDE="{DEVICE_OVERRIDE}" is invalid; falling back to "auto".')
    device_override_normalized = "auto"

cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

def _resolve_cuda_indices():
    if not CUDA_DEVICE_INDICES:
        return None
    valid = []
    max_idx = torch.cuda.device_count() - 1
    for idx in CUDA_DEVICE_INDICES:
        if isinstance(idx, (int, np.integer)) and 0 <= int(idx) <= max_idx:
            valid.append(int(idx))
        else:
            print(f"Warning: CUDA_DEVICE_INDICES contains invalid index {idx}; skipping.")
    if not valid:
        print("Warning: CUDA_DEVICE_INDICES is set but no valid indices remain; falling back to default selection.")
        return None
    return valid

selected_gpu_indices = None
selected_gpu_index = None
if device_override_normalized == "cuda":
    if cuda_available:
        selected_gpu_indices = _resolve_cuda_indices()
        if selected_gpu_indices is None:
            selected_gpu_index = min(PREFERRED_GPU_INDEX, torch.cuda.device_count() - 1)
            selected_gpu_indices = [selected_gpu_index]
        else:
            selected_gpu_index = selected_gpu_indices[0]
        torch.cuda.set_device(selected_gpu_index)
        active_device = torch.device(f"cuda:{selected_gpu_index}")
        selected_accelerator = "cuda"
    else:
        print("Warning: CUDA requested but not available; using CPU.")
        active_device = torch.device("cpu")
        selected_accelerator = "cpu"
elif device_override_normalized == "mps":
    if mps_available:
        active_device = torch.device("mps")
        selected_accelerator = "mps"
    else:
        print("Warning: MPS requested but not available; using CPU.")
        active_device = torch.device("cpu")
        selected_accelerator = "cpu"
elif device_override_normalized == "cpu":
    active_device = torch.device("cpu")
    selected_accelerator = "cpu"
else:
    if cuda_available:
        selected_gpu_indices = _resolve_cuda_indices()
        if selected_gpu_indices is None:
            selected_gpu_index = min(PREFERRED_GPU_INDEX, torch.cuda.device_count() - 1)
            selected_gpu_indices = [selected_gpu_index]
        else:
            selected_gpu_index = selected_gpu_indices[0]
        torch.cuda.set_device(selected_gpu_index)
        active_device = torch.device(f"cuda:{selected_gpu_index}")
        selected_accelerator = "cuda"
    elif mps_available:
        active_device = torch.device("mps")
        selected_accelerator = "mps"
    else:
        active_device = torch.device("cpu")
        selected_accelerator = "cpu"

print(f'Torch {torch.__version__} | DEVICE_OVERRIDE: {DEVICE_OVERRIDE} | Device: {active_device}')
if selected_accelerator == "cuda":
    print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
    if selected_gpu_indices and len(selected_gpu_indices) > 1:
        print("CUDA_DEVICE_INDICES:", selected_gpu_indices)
elif selected_accelerator == "mps":
    print("Using Apple Silicon MPS accelerator.")
else:
    print("Using CPU backend.")


# Helper: wrap model for multi-GPU when CUDA_DEVICE_INDICES has >1 entries
def maybe_wrap_dataparallel(model):
    if selected_accelerator == "cuda" and selected_gpu_indices and len(selected_gpu_indices) > 1:
        return nn.DataParallel(model, device_ids=selected_gpu_indices)
    return model


## Block 3: Optimizer Dynamo Patch

Applies a compatibility patch for Optimizer methods to avoid torch._dynamo issues on some environments.


In [ ]:
# Compatibility workaround: some Torch builds fail while importing torch._dynamo
# via @torch._disable_dynamo wrappers used in optimizer methods.
def patch_optimizer_dynamo_wrappers():
    optimizer_class = torch.optim.Optimizer
    method_names = ["add_param_group", "zero_grad", "state_dict", "load_state_dict"]
    for method_name in method_names:
        wrapped_method = getattr(optimizer_class, method_name, None)
        raw_method = getattr(wrapped_method, "__wrapped__", None)
        if raw_method is not None and not hasattr(raw_method, "__dynamo_disable"):
            raw_method.__dynamo_disable = raw_method

patch_optimizer_dynamo_wrappers()


## Block 4: Adam Step Patch

Unwraps Adam/AdamW step wrappers to prevent runtime failures caused by problematic dynamo hooks.


In [ ]:
# Additional compatibility patch: unwrap Adam/AdamW step to avoid `_use_grad`
# importing torch._dynamo at runtime on affected builds.
def patch_adam_step_wrappers():
    for optimizer_class in [torch.optim.Adam, torch.optim.AdamW]:
        step_fn = getattr(optimizer_class, "step", None)
        while hasattr(step_fn, "__wrapped__"):
            step_fn = step_fn.__wrapped__
        optimizer_class.step = step_fn
        # Prevent Optimizer._patch_step_function from wrapping this again.
        optimizer_class.step.hooked = True

patch_adam_step_wrappers()


## Block 5: Optional Package Setup

Checks for required packages and installs missing ones, then imports Optuna, tqdm, matplotlib, and torch.nn.functional.


In [ ]:
# Install runtime dependencies (optional ones may fail gracefully on some machines).
PACKAGE_IMPORT_NAME_MAP = {
    "scikit-learn": "sklearn",
}
OPTIONAL_PIP_PACKAGES = {"xformers", "ipywidgets", "torchmetrics"}
optional_install_failures = {}

def ensure_typing_extensions_features():
    required_attributes = ("TypeIs", "TypeAliasType")
    try:
        import typing_extensions as typing_extensions_module
    except Exception:
        typing_extensions_module = None

    has_required_attributes = (
        typing_extensions_module is not None
        and all(hasattr(typing_extensions_module, attr_name) for attr_name in required_attributes)
    )
    if has_required_attributes:
        return

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "typing_extensions"])
    import importlib
    typing_extensions_module = importlib.import_module("typing_extensions")
    typing_extensions_module = importlib.reload(typing_extensions_module)

    missing_attributes = [
        attr_name for attr_name in required_attributes if not hasattr(typing_extensions_module, attr_name)
    ]
    if missing_attributes:
        raise ImportError(
            "typing_extensions is missing required attributes after upgrade: "
            + ", ".join(missing_attributes)
        )
    print("[deps] upgraded typing_extensions for aequitas/altair compatibility.")


ensure_typing_extensions_features()

for package_name in AUTO_INSTALL_PACKAGES:
    import_name = PACKAGE_IMPORT_NAME_MAP.get(package_name, package_name)
    try:
        __import__(import_name)
        continue
    except Exception:
        pass

    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        __import__(import_name)
        print(f"[deps] installed: {package_name}")
    except Exception as install_error:
        if package_name in OPTIONAL_PIP_PACKAGES:
            optional_install_failures[package_name] = str(install_error)
            print(f"[deps] optional package unavailable, continuing: {package_name} | {install_error}")
        else:
            raise

if optional_install_failures:
    print("Optional dependency install failures (safe to continue unless you explicitly use them):")
    for package_name, failure_text in optional_install_failures.items():
        print(f" - {package_name}: {failure_text}")

import optuna
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
import types



## Block 6: Dense Head Baseline

Confirms this baseline notebook uses a plain dense classification head and no spiking components.


In [ ]:
# Pure FT-Transformer baseline: no extra head-specific libraries are used.
print("Using pure FT-Transformer dense head baseline (no SNN head).")


## Block 7: Experiment Configuration

Defines paths, random seed, optimization settings, logging schemas, and export configuration.


In [ ]:
# ---------------- Config ----------------
# Reproducibility and runtime summary
for _path_dir in [RUN_OUTPUT_ROOT, OPTUNA_DIR, CSV_EXPORT_DIR, MODEL_EXPORT_DIR, PAPER_EXPORT_DIR]:
    _path_dir.mkdir(parents=True, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {active_device}")
if active_device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
elif active_device.type == "mps":
    print("MPS backend active.")
else:
    print("CPU backend active.")

print(f"Experiment tag : {EXPERIMENT_TAG}")
print(f"Dataset path   : {DATA_CSV_PATH}")
print(f"Run root       : {RUN_OUTPUT_ROOT}")
print(f"Optuna dir     : {OPTUNA_DIR}")
print(f"CSV dir        : {CSV_EXPORT_DIR}")
print(f"Model dir      : {MODEL_EXPORT_DIR}")
print(f"Paper dir      : {PAPER_EXPORT_DIR}")
print(f"Optuna DB      : {OPTUNA_STORAGE_PATH}")
print(f"Trials CSV     : {TRIALS_CSV}")
print(f"Tuning budget  : {TUNING_TRIALS} trials | {TUNING_EPOCHS} epochs per trial | Final retrain={FINAL_EPOCHS}")


## Block 8: Dataset Loading

Resolves the dataset path and loads the selected CSV into a dataframe.


In [ ]:
# ---------------- Data prep ----------------
def resolve_dataset_path(raw_path: Path) -> Path:
    """Resolve dataset path across local and Colab-style runtimes.

    Priority:
    1) Explicit raw path / file name in current working dir
    2) Configured DATASET_DIR
    3) Optional env overrides: BAF_DATASET_PATH or BAF_DATASET_DIR
    4) Common Colab locations
    """
    dataset_file_name = Path(DATASET_FILE_NAME).name

    env_dataset_path = os.environ.get("BAF_DATASET_PATH", "").strip()
    env_dataset_dir = os.environ.get("BAF_DATASET_DIR", "").strip()

    candidate_paths = [
        Path(raw_path),
        Path(dataset_file_name),
        Path(DATASET_DIR) / dataset_file_name,
    ]

    if env_dataset_path:
        candidate_paths.append(Path(env_dataset_path))
    if env_dataset_dir:
        candidate_paths.append(Path(env_dataset_dir) / dataset_file_name)

    # Common Colab / Drive paths
    candidate_paths.extend([
        Path("/content") / dataset_file_name,
        Path("/content/baf-datasets") / dataset_file_name,
        Path("/content/drive/MyDrive/baf-datasets") / dataset_file_name,
        Path("/content/drive/MyDrive/Colab Notebooks/baf-datasets") / dataset_file_name,
    ])

    # Dedupe while preserving order
    seen = set()
    ordered_candidates = []
    for candidate_path in candidate_paths:
        candidate_path = Path(candidate_path)
        candidate_key = str(candidate_path)
        if candidate_key in seen:
            continue
        seen.add(candidate_key)
        ordered_candidates.append(candidate_path)

    for candidate_path in ordered_candidates:
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(
        "Dataset not found. Tried paths:\n - " + "\n - ".join(str(path) for path in ordered_candidates)
    )



DATA_CSV_PATH = resolve_dataset_path(DATA_CSV_PATH)
DATASET_DIR = DATA_CSV_PATH.parent
full_df = pd.read_csv(DATA_CSV_PATH)
print(f"Resolved dataset path: {DATA_CSV_PATH}")

## Block 9: Target and Time Split

Finds target column, validates month field, parses month values, and creates train/valid vs test time splits.


In [ ]:
# Detect the target column from common names.
for candidate_target_col in TARGET_CANDIDATE_COLUMNS:
    if candidate_target_col in full_df.columns:
        target_col = candidate_target_col
        break
else:
    raise ValueError(f"Set target column name. Tried: {TARGET_CANDIDATE_COLUMNS}")

# Time split by month: <= TRAIN_MONTH_MAX for train/valid, >= TEST_MONTH_MIN for test.
if MONTH_COLUMN not in full_df.columns:
    raise ValueError(f"Expected a '{MONTH_COLUMN}' column for time-based split.")


def parse_int_from_value(value):
    try:
        return int(re.findall(r"-?\d+", str(value))[0])
    except Exception:
        return np.nan


month_values = full_df[MONTH_COLUMN].map(parse_int_from_value)
if month_values.isna().any():
    raise ValueError(f"Could not parse some '{MONTH_COLUMN}' values as integers.")
full_df = full_df.assign(**{MONTH_COLUMN: month_values.astype(int)})

train_valid_df = full_df[full_df[MONTH_COLUMN] <= TRAIN_MONTH_MAX].copy()
test_df = full_df[full_df[MONTH_COLUMN] >= TEST_MONTH_MIN].copy()
if len(train_valid_df) == 0 or len(test_df) == 0:
    raise ValueError(
        f"Time split produced empty train/valid or test set. Check '{MONTH_COLUMN}' range in CSV."
    )

## Block 10: Feature Typing

Builds feature list and separates columns into categorical and continuous feature groups.


In [ ]:
# Build feature lists.
feature_cols = [
    column_name
    for column_name in full_df.columns
    if column_name != target_col
    and column_name != MONTH_COLUMN
    and not any(fragment in column_name.lower() for fragment in DROP_FEATURE_NAME_FRAGMENTS)
]

categorical_cols = [
    column_name for column_name in feature_cols
    if str(full_df[column_name].dtype) in ("object", "category")
]
for column_name in feature_cols:
    if (
        column_name not in categorical_cols
        and pd.api.types.is_integer_dtype(full_df[column_name])
        and full_df[column_name].nunique() <= INT_AS_CATEGORICAL_MAX_UNIQUE
    ):
        categorical_cols.append(column_name)

continuous_cols = [
    column_name
    for column_name in feature_cols
    if column_name not in categorical_cols and pd.api.types.is_numeric_dtype(full_df[column_name])
]

## Block 11: Fairness Group Definition

Defines helper functions to derive age-based protected group masks (>=50 vs <50).


In [ ]:
# Fairness groups for Aequitas audits (age, income, employment_status).
try:
    from aequitas.group import Group as AequitasGroup
    from aequitas.bias import Bias as AequitasBias
except Exception as aequitas_import_error:
    AequitasGroup = None
    AequitasBias = None
    _AEQUITAS_IMPORT_ERROR = aequitas_import_error
else:
    _AEQUITAS_IMPORT_ERROR = None


def _first_existing_column(dataframe: pd.DataFrame, candidate_columns):
    for column_name in candidate_columns:
        if column_name in dataframe.columns:
            return column_name
    return None


def extract_numeric_age(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    numeric_text = series.astype(str).str.extract(r"(-?\d+)", expand=False)
    return pd.to_numeric(numeric_text, errors="coerce")


def extract_numeric_feature(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    numeric_text = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.extract(r"(-?\d+\.?\d*)", expand=False)
    )
    return pd.to_numeric(numeric_text, errors="coerce")


def make_age_group_values(dataframe: pd.DataFrame):
    age_column_name = _first_existing_column(dataframe, AGE_GROUP_COLUMNS)
    if age_column_name is None:
        return pd.Series("Unknown", index=dataframe.index, dtype=object)

    parsed_age = extract_numeric_age(dataframe[age_column_name])
    age_labels = pd.Series(
        np.where(parsed_age >= AGE_THRESHOLD, f">={AGE_THRESHOLD}", f"<{AGE_THRESHOLD}"),
        index=dataframe.index,
    )
    age_labels[parsed_age.isna()] = "Unknown"
    return age_labels.astype(str)


def fit_income_group_bin_edges(train_dataframe: pd.DataFrame):
    income_column_name = _first_existing_column(train_dataframe, INCOME_GROUP_COLUMNS)
    if income_column_name is None:
        return None

    parsed_income = extract_numeric_feature(train_dataframe[income_column_name]).dropna()
    if parsed_income.empty or parsed_income.nunique() < 4:
        return None

    quantiles = parsed_income.quantile(INCOME_GROUP_QUANTILES).to_numpy(dtype=float)
    bin_edges = np.unique(quantiles)
    return bin_edges if len(bin_edges) >= 2 else None


def make_income_group_values(dataframe: pd.DataFrame, income_bin_edges=None):
    income_column_name = _first_existing_column(dataframe, INCOME_GROUP_COLUMNS)
    if income_column_name is None:
        return pd.Series("Unknown", index=dataframe.index, dtype=object)

    parsed_income = extract_numeric_feature(dataframe[income_column_name])
    if income_bin_edges is not None:
        bin_edges = np.unique(np.asarray(income_bin_edges, dtype=float))
        if len(bin_edges) >= 2:
            income_bins = pd.cut(parsed_income, bins=bin_edges, include_lowest=True, duplicates="drop")
            income_labels = income_bins.astype(str).replace("nan", "Unknown")
            return income_labels.fillna("Unknown").astype(str)

    income_labels = dataframe[income_column_name].astype(str).str.strip()
    income_labels = income_labels.replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})
    return income_labels.fillna("Unknown").astype(str)


def make_employment_status_group_values(dataframe: pd.DataFrame):
    employment_column_name = _first_existing_column(dataframe, EMPLOYMENT_STATUS_GROUP_COLUMNS)
    if employment_column_name is None:
        return pd.Series("Unknown", index=dataframe.index, dtype=object)

    employment_labels = dataframe[employment_column_name].astype(str).str.strip()
    employment_labels = employment_labels.replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})
    return employment_labels.fillna("Unknown").astype(str)


def build_sensitive_attribute_frame(dataframe: pd.DataFrame, income_bin_edges=None):
    return pd.DataFrame(
        {
            "age_group": make_age_group_values(dataframe),
            "income_group": make_income_group_values(dataframe, income_bin_edges=income_bin_edges),
            "employment_status_group": make_employment_status_group_values(dataframe),
        },
        index=dataframe.index,
    )


## Block 12: Train/Valid Split

Splits months 0-5 into train and validation sets and computes group masks for all splits.


In [ ]:
# Split train/valid inside months <= TRAIN_MONTH_MAX.
from sklearn.model_selection import train_test_split as sklearn_train_test_split

train_df, valid_df = sklearn_train_test_split(
    train_valid_df,
    test_size=VALID_SPLIT_RATIO,
    stratify=train_valid_df[target_col],
    random_state=RANDOM_SEED,
)

income_group_bin_edges = fit_income_group_bin_edges(train_df)
train_sensitive_attributes_df = build_sensitive_attribute_frame(train_df, income_bin_edges=income_group_bin_edges)
valid_sensitive_attributes_df = build_sensitive_attribute_frame(valid_df, income_bin_edges=income_group_bin_edges)
test_sensitive_attributes_df = build_sensitive_attribute_frame(test_df, income_bin_edges=income_group_bin_edges)

## Block 13: Category Vocab and Stats

Locks categorical vocab on train-valid data and computes continuous feature means for imputation.


In [ ]:
# Lock categorical vocab on TRAIN(0-5) and reserve one UNK index per feature.
categorical_levels = {}
categorical_cardinalities = {}
for column_name in categorical_cols:
    known_levels = train_valid_df[column_name].astype("category").cat.categories
    categorical_levels[column_name] = list(known_levels)
    categorical_cardinalities[column_name] = len(known_levels) + 1  # +1 for UNK

cat_cardinalities = [categorical_cardinalities[column_name] for column_name in categorical_cols] if categorical_cols else None

# Train means for continuous features (used for imputation everywhere).
continuous_train_mean = train_valid_df[continuous_cols].astype(float).mean() if continuous_cols else None


## Block 14: Encoding Functions

Defines categorical/continuous encoders and a helper to build tensor tuples for each split.


In [ ]:
def encode_categorical_features(dataframe_part: pd.DataFrame) -> torch.Tensor | None:
    if not categorical_cols:
        return None
    encoded_columns = []
    for column_name in categorical_cols:
        raw_codes = pd.Categorical(dataframe_part[column_name], categories=categorical_levels[column_name]).codes
        unknown_index = categorical_cardinalities[column_name] - 1
        safe_codes = np.where(raw_codes == -1, unknown_index, raw_codes).astype("int64")
        encoded_columns.append(safe_codes)
    stacked = np.stack(encoded_columns, axis=1) if encoded_columns else None
    return torch.as_tensor(stacked, dtype=torch.long) if stacked is not None else None


def encode_continuous_features(dataframe_part: pd.DataFrame) -> torch.Tensor | None:
    if not continuous_cols:
        return None
    numeric_values = dataframe_part[continuous_cols].astype(float).copy()
    numeric_values = numeric_values.fillna(continuous_train_mean)
    return torch.as_tensor(numeric_values.values, dtype=torch.float32)


def build_split_tensors(dataframe_part: pd.DataFrame):
    x_num = encode_continuous_features(dataframe_part)
    x_cat = encode_categorical_features(dataframe_part)
    y_target = torch.as_tensor(dataframe_part[target_col].values, dtype=torch.float32).view(-1, 1)
    return x_num, x_cat, y_target


## Block 15: Tensor Construction and Normalization

Builds tensors for train/valid/test and normalizes continuous features using train statistics.


In [ ]:
# Build tensors for each split.
x_num_train, x_cat_train, y_train = build_split_tensors(train_df)
x_num_valid, x_cat_valid, y_valid = build_split_tensors(valid_df)
x_num_test, x_cat_test, y_test = build_split_tensors(test_df)

# Normalize continuous features using TRAIN stats only.
if x_num_train is not None:
    train_mean = x_num_train.mean(0, keepdim=True)
    train_std = x_num_train.std(0, keepdim=True).clamp_min(1e-6)
    x_num_train = (x_num_train - train_mean) / train_std
    if x_num_valid is not None:
        x_num_valid = (x_num_valid - train_mean) / train_std
    if x_num_test is not None:
        x_num_test = (x_num_test - train_mean) / train_std


## Block 16: Device Batch Helper

Defines helper to move a batch (numeric, categorical, target) tensors to the active device.


In [ ]:
def move_batch_tensors_to_device(x_num, x_cat, y_target):
    batch_parts = []
    if x_num is not None:
        batch_parts.append(x_num.to(active_device))
    if x_cat is not None:
        batch_parts.append(x_cat.to(active_device))
    batch_parts.append(y_target.to(active_device))
    return batch_parts


# Number of numeric features used by tokenizer.
num_numeric_features = 0 if x_num_train is None else x_num_train.shape[1]


## Block 17: Feature Count and Index Checks

Computes numeric feature count and validates categorical indices against embedding cardinalities.


In [ ]:
# Sanity check: ensure categorical codes never exceed embedding cardinalities.
if x_cat_train is not None:
    for column_index, column_name in enumerate(categorical_cols):
        max_seen_code = int(max(
            (int(x_cat_train[:, column_index].max().item()) if x_cat_train.numel() else 0),
            (int(x_cat_valid[:, column_index].max().item()) if x_cat_valid is not None and x_cat_valid.numel() else 0),
            (int(x_cat_test[:, column_index].max().item()) if x_cat_test is not None and x_cat_test.numel() else 0),
        ))
        assert max_seen_code < categorical_cardinalities[column_name], (
            f"Index overflow in '{column_name}': {max_seen_code} >= {categorical_cardinalities[column_name]}"
        )


## Block 18: Feature Tokenizer

Defines FeatureTokenizer that converts numeric and categorical features into token embeddings with a CLS token.


In [ ]:
# ---------------- FT-Transformer baseline ----------------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features: int, cat_feature_cardinalities, d_token: int):
        super().__init__()
        self.n_num_features = n_num_features
        self.cat_feature_cardinalities = cat_feature_cardinalities or []
        self.d_token = d_token

        if self.n_num_features > 0:
            self.num_weight = nn.Parameter(torch.randn(self.n_num_features, d_token) * 0.02)
            self.num_bias = nn.Parameter(torch.zeros(self.n_num_features, d_token))
        else:
            self.register_parameter("num_weight", None)
            self.register_parameter("num_bias", None)

        self.cat_embeds = nn.ModuleList([
            nn.Embedding(cardinality, d_token)
            for cardinality in self.cat_feature_cardinalities
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)

    def forward(self, x_num, x_cat):
        batch_size = x_num.size(0) if x_num is not None else x_cat.size(0)
        token_blocks = []

        if x_num is not None:
            token_blocks.append(
                x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0)
            )

        if x_cat is not None and len(self.cat_feature_cardinalities) > 0:
            cat_embeddings = [embedding(x_cat[:, idx]) for idx, embedding in enumerate(self.cat_embeds)]
            token_blocks.append(torch.stack(cat_embeddings, dim=1))

        if not token_blocks:
            raise ValueError("No features to tokenize")

        all_tokens = torch.cat(token_blocks, dim=1)
        all_tokens = torch.cat([self.cls_token.expand(batch_size, 1, -1), all_tokens], dim=1)
        return all_tokens


## Block 19: Dense Head

Defines the plain linear classification head used by the FT-Transformer baseline.


In [ ]:
class DenseHead(nn.Module):
    """Simple linear classification head over the CLS embedding."""

    def __init__(self, d_token):
        super().__init__()
        self.readout = nn.Linear(d_token, 1)
        nn.init.zeros_(self.readout.bias)

    def forward(self, cls_embedding):
        return self.readout(cls_embedding)


## Block 20: FTTransformer Model

Defines the FT-Transformer baseline with tokenizer, transformer encoder, and plain dense head.


In [ ]:
class FTTransformer(nn.Module):
    def __init__(
        self,
        n_num_features,
        cat_feature_cardinalities,
        d_token=FT_DEFAULT_D_TOKEN,
        n_blocks=FT_DEFAULT_N_BLOCKS,
        n_heads=FT_DEFAULT_N_HEADS,
        ffn_hidden=FT_DEFAULT_FFN_HIDDEN,
        dropout=FT_DEFAULT_DROPOUT,
        **unused_head_kwargs,
    ):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, cat_feature_cardinalities, d_token)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=ffn_hidden,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.head = DenseHead(d_token)

    def forward(self, x_num, x_cat):
        token_sequence = self.tokenizer(x_num, x_cat)
        encoded_tokens = self.encoder(token_sequence)
        cls_embedding = encoded_tokens[:, 0, :]
        return self.head(cls_embedding)


## Block 21: Recall at FPR Metric

Defines helper to pick the best recall under a specified FPR cap and return recall/threshold/FPR.


In [ ]:
# The model outputs a probability score for each sample, and we try many threshold cutoffs (alarm sensitivity levels) to convert scores into positive/negative predictions.
# For each threshold we compute FPR/recall, then choose the threshold with the highest recall that still satisfies the FPR cap.

# ---------------- Metrics helpers ----------------
def recall_at_fpr_cap(y_true_tensor, y_probabilities, cap=FPR_CAP):
    y_true_np = y_true_tensor.cpu().numpy().ravel()
    fpr_values, tpr_values, thresholds = roc_curve(y_true_np, y_probabilities)
    under_cap_mask = fpr_values <= cap
    if not np.any(under_cap_mask):
        return 0.0, DEFAULT_THRESHOLD, 1.0

    best_index_within_cap = np.argmax(tpr_values[under_cap_mask])
    return (
        tpr_values[under_cap_mask][best_index_within_cap],
        thresholds[under_cap_mask][best_index_within_cap],
        fpr_values[under_cap_mask][best_index_within_cap],
    )
# False Positive Rate (FPR) = FP / (FP + TN): among actual negatives, how many were wrongly predicted positive.
# Fairness ratio at this threshold: smoothed min(group FPR)/max(group FPR) in [0, 1]; 1.0 is ideal.

## Block 22: Aequitas Fairness Audit Helpers

Defines Aequitas-based predictive-equality auditing across `age`, `income`, and `employment_status` at a fixed decision threshold.

**What is reported:**
- We compute Aequitas group-level FPR disparities (relative to the major group) for each sensitive attribute.
- The notebook logs a bounded summary score in `[0, 1]` by converting each disparity to a parity score `min(d, 1/d)` and taking the worst case across groups/attributes (`1.0` is ideal).


In [ ]:
def _aequitas_get_crosstabs_df(audit_df: pd.DataFrame, attr_cols):
    group_auditor = AequitasGroup()
    crosstab_result = group_auditor.get_crosstabs(audit_df, attr_cols=attr_cols)
    return crosstab_result[0] if isinstance(crosstab_result, tuple) else crosstab_result


def _aequitas_get_major_group_disparity_df(crosstab_df: pd.DataFrame, audit_df: pd.DataFrame):
    bias_auditor = AequitasBias()
    try:
        disparity_result = bias_auditor.get_disparity_major_group(
            crosstab_df,
            original_df=audit_df,
            mask_significance=False,
        )
    except TypeError:
        try:
            disparity_result = bias_auditor.get_disparity_major_group(
                crosstab_df,
                df=audit_df,
                mask_significance=False,
            )
        except TypeError:
            disparity_result = bias_auditor.get_disparity_major_group(crosstab_df, audit_df)
    return disparity_result[0] if isinstance(disparity_result, tuple) else disparity_result


def aequitas_fpr_parity_summary_at_threshold(y_true, y_probabilities, sensitive_attribute_df, threshold):
    """Aequitas predictive-equality audit using FPR disparity to the major group.

    Degenerate attributes (fewer than 2 distinct observed groups in the audited split)
    are flagged and excluded from the overall min-parity aggregation.
    """
    if AequitasGroup is None or AequitasBias is None:
        raise RuntimeError(
            "Aequitas is required for fairness evaluation in this notebook. "
            f"Original import error: {_AEQUITAS_IMPORT_ERROR}"
        )

    y_true_int = np.asarray(y_true).ravel().astype(int)
    y_pred_int = (np.asarray(y_probabilities) >= float(threshold)).astype(int)

    audit_df = sensitive_attribute_df.copy()
    audit_df["score"] = y_pred_int.astype(float)
    audit_df["label_value"] = y_true_int

    attr_cols = [attribute_name for attribute_name in AEQUITAS_FAIRNESS_ATTRIBUTES if attribute_name in audit_df.columns]
    if not attr_cols:
        return float("nan"), {}, pd.DataFrame()

    # Count observed groups per attribute before Aequitas, so we can flag degenerate cases
    # (e.g., a missing source column collapsed to a single 'Unknown' group).
    attribute_group_counts = {}
    for attribute_name in attr_cols:
        attribute_values = audit_df[attribute_name].astype(str).fillna("Unknown")
        attribute_group_counts[attribute_name] = int(pd.Series(attribute_values).nunique(dropna=False))

    crosstab_df = _aequitas_get_crosstabs_df(audit_df, attr_cols=attr_cols)
    disparity_df = _aequitas_get_major_group_disparity_df(crosstab_df, audit_df)

    if isinstance(disparity_df, pd.DataFrame) and not disparity_df.empty and "attribute_name" in disparity_df.columns:
        disparity_df = disparity_df.copy()
        disparity_df["attribute_n_distinct_groups"] = disparity_df["attribute_name"].map(attribute_group_counts)
        disparity_df["attribute_is_degenerate"] = (
            pd.to_numeric(disparity_df["attribute_n_distinct_groups"], errors="coerce")
            .fillna(0)
            .lt(2)
            .astype(int)
        )

    per_attribute_scores = {}
    for attribute_name in attr_cols:
        group_count = int(attribute_group_counts.get(attribute_name, 0))
        if group_count < 2:
            per_attribute_scores[attribute_name] = float("nan")
            continue

        attribute_rows = disparity_df[disparity_df["attribute_name"] == attribute_name].copy()
        if "fpr_disparity" not in attribute_rows.columns or attribute_rows.empty:
            per_attribute_scores[attribute_name] = float("nan")
            continue

        disparities = pd.to_numeric(attribute_rows["fpr_disparity"], errors="coerce").to_numpy(dtype=float)
        if disparities.size == 0:
            per_attribute_scores[attribute_name] = float("nan")
            continue

        parity_scores = np.full(disparities.shape, np.nan, dtype=float)
        finite_positive_mask = np.isfinite(disparities) & (disparities > 0.0)
        parity_scores[finite_positive_mask] = np.minimum(
            disparities[finite_positive_mask],
            1.0 / disparities[finite_positive_mask],
        )
        parity_scores[np.isinf(disparities) | (disparities == 0.0)] = 0.0

        valid_parity_scores = parity_scores[np.isfinite(parity_scores)]
        if valid_parity_scores.size == 0:
            per_attribute_scores[attribute_name] = float("nan")
            continue

        per_attribute_scores[attribute_name] = float(np.clip(np.min(valid_parity_scores), 0.0, 1.0))

    finite_scores = [score for score in per_attribute_scores.values() if np.isfinite(score)]
    overall_score = float(min(finite_scores)) if finite_scores else float("nan")
    return overall_score, per_attribute_scores, disparity_df


def aequitas_attribute_metadata_by_attribute(disparity_df: pd.DataFrame):
    """Extract per-attribute degeneracy metadata from an Aequitas disparity dataframe.

    Returns:
        (n_distinct_groups_by_attr, is_degenerate_by_attr)
    """
    if not isinstance(disparity_df, pd.DataFrame) or disparity_df.empty or "attribute_name" not in disparity_df.columns:
        return {}, {}

    attr_rows = disparity_df[[
        column_name for column_name in [
            "attribute_name",
            "attribute_n_distinct_groups",
            "attribute_is_degenerate",
        ] if column_name in disparity_df.columns
    ]].copy()
    if "attribute_name" not in attr_rows.columns:
        return {}, {}

    attr_rows = attr_rows.drop_duplicates(subset=["attribute_name"], keep="first")

    if "attribute_n_distinct_groups" in attr_rows.columns:
        n_distinct_groups_by_attr = {
            str(attribute_name): int(value)
            for attribute_name, value in zip(
                attr_rows["attribute_name"],
                pd.to_numeric(attr_rows["attribute_n_distinct_groups"], errors="coerce").fillna(-1).astype(int),
            )
            if int(value) >= 0
        }
    else:
        n_distinct_groups_by_attr = {}

    if "attribute_is_degenerate" in attr_rows.columns:
        is_degenerate_by_attr = {
            str(attribute_name): int(value)
            for attribute_name, value in zip(
                attr_rows["attribute_name"],
                pd.to_numeric(attr_rows["attribute_is_degenerate"], errors="coerce").fillna(-1).astype(int),
            )
            if int(value) in (0, 1)
        }
    else:
        is_degenerate_by_attr = {}

    return n_distinct_groups_by_attr, is_degenerate_by_attr


## Block 23: Objective Setup

Resets CSV logs and defines the single-study FT-Transformer baseline optimization workflow used below.

### Implemented Hyper-Optimization Flow

- One Optuna study tunes FT-Transformer backbone size, dropout, learning rate, and batch size.
- The dense linear head is fixed; there is no secondary head search or refinement stage.
- The same train/valid split is reused for every trial.
- Test split is only used for reporting/logging snapshots and final evaluation.
- Optuna pruner + early stopping are enabled.


In [ ]:
# ---------------- Objective ----------------
def ensure_csv_header_compatibility(csv_path: Path, expected_fields):
    if not csv_path.exists():
        return
    with open(csv_path, newline="") as csv_file:
        reader = csv.reader(csv_file)
        existing_header = next(reader, [])
    if list(existing_header) == list(expected_fields):
        return
    archived_path = csv_path.with_suffix(csv_path.suffix + ".legacy")
    csv_path.rename(archived_path)
    print(f"Archived incompatible prior log to {archived_path}")


if RESET_TRIAL_LOG_CSVS:
    print("Resetting trial/best/fairness CSV logs for a fresh export.")
    TRIALS_CSV.unlink(missing_ok=True)
    BEST_CSV.unlink(missing_ok=True)
    FAIR_CSV.unlink(missing_ok=True)
else:
    print("Keeping existing trial/best/fairness CSV logs (resume mode).")
    ensure_csv_header_compatibility(TRIALS_CSV, TRIAL_LOG_FIELDS)
    ensure_csv_header_compatibility(BEST_CSV, TRIAL_LOG_FIELDS)
    ensure_csv_header_compatibility(FAIR_CSV, FAIR_LOG_FIELDS)


## Block 24: DataLoader Builder

Defines helper function to create device-ready DataLoaders for train/valid/test tensors.


In [ ]:
def build_loader(x_num, x_cat, y_target, batch_size, shuffle=False):
    return DataLoader(
        TensorDataset(*move_batch_tensors_to_device(x_num, x_cat, y_target)),
        batch_size=batch_size,
        shuffle=shuffle,
    )


## Block 25: Tuning Objective Functions

Defines reusable training/evaluation helpers and the single-study Optuna objective for the FT-Transformer dense baseline.


In [ ]:
def append_csv_row(csv_path: Path, fieldnames, row_dict: dict):
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)


def safe_binary_metric(metric_fn, y_true, y_probabilities):
    try:
        return float(metric_fn(y_true, y_probabilities))
    except ValueError:
        return float("nan")


def default_ft_params():
    return {
        "d_token": int(FT_DEFAULT_D_TOKEN),
        "n_blocks": int(FT_DEFAULT_N_BLOCKS),
        "n_heads": int(FT_DEFAULT_N_HEADS),
        "ffn_hidden": int(FT_DEFAULT_FFN_HIDDEN),
        "dropout": float(FT_DEFAULT_DROPOUT),
    }


def extract_ft_params(source_params: dict):
    ft = default_ft_params()
    for key in ["d_token", "n_blocks", "n_heads", "ffn_hidden", "dropout"]:
        if key in source_params:
            ft[key] = source_params[key]
    ft["d_token"] = int(ft["d_token"])
    ft["n_blocks"] = int(ft["n_blocks"])
    ft["n_heads"] = int(ft["n_heads"])
    ft["ffn_hidden"] = int(ft["ffn_hidden"])
    ft["dropout"] = float(ft["dropout"])
    return ft


def local_choice_window(all_choices, center_value, radius=1):
    choices = sorted(set(all_choices))
    if center_value in choices:
        center_index = choices.index(center_value)
    else:
        center_index = min(range(len(choices)), key=lambda idx: abs(float(choices[idx]) - float(center_value)))
    left = max(0, center_index - int(radius))
    right = min(len(choices), center_index + int(radius) + 1)
    return choices[left:right]


def local_float_bounds(center, lower, upper, fraction=0.30, min_pad=1e-3):
    span = max(float(upper) - float(lower), 1e-9)
    pad = max(span * float(fraction) * 0.5, float(min_pad))
    lo = max(float(lower), float(center) - pad)
    hi = min(float(upper), float(center) + pad)
    if hi <= lo:
        lo, hi = float(lower), float(upper)
    return lo, hi


def local_log_bounds(center, lower, upper, log10_radius=0.35):
    center = float(np.clip(center, lower, upper))
    ratio = 10 ** float(log10_radius)
    lo = max(float(lower), center / ratio)
    hi = min(float(upper), center * ratio)
    if hi <= lo:
        lo, hi = float(lower), float(upper)
    return lo, hi


def run_single_trial(
    trial: optuna.Trial,
    stage_label: str,
    ft_params: dict,
    learning_rate: float,
    batch_size: int,
    epochs: int,
):
    ft_params = extract_ft_params(ft_params)
    learning_rate = float(learning_rate)
    batch_size = int(batch_size)

    model = FTTransformer(
        num_numeric_features,
        cat_cardinalities,
        d_token=ft_params["d_token"],
        n_blocks=ft_params["n_blocks"],
        n_heads=ft_params["n_heads"],
        ffn_hidden=ft_params["ffn_hidden"],
        dropout=ft_params["dropout"],
    ).to(active_device)
    model = maybe_wrap_dataparallel(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loader = build_loader(x_num_train, x_cat_train, y_train, batch_size, shuffle=True)
    valid_loader = build_loader(
        x_num_valid,
        x_cat_valid,
        y_valid,
        batch_size * VALID_TEST_BATCH_MULTIPLIER,
        shuffle=False,
    )
    test_loader = None
    if LOG_TEST_DURING_TUNING:
        test_loader = build_loader(
            x_num_test,
            x_cat_test,
            y_test,
            batch_size * VALID_TEST_BATCH_MULTIPLIER,
            shuffle=False,
        )

    def predict_probabilities(data_loader):
        model.eval()
        probability_chunks = []
        with torch.no_grad():
            for *features, _ in data_loader:
                probability_chunks.append(torch.sigmoid(model(*features)).squeeze(1).cpu())
        return torch.cat(probability_chunks).numpy()

    best_valid_recall = -1.0
    best_trial_row = None
    best_model_state = None
    best_validation_threshold = DEFAULT_THRESHOLD
    no_improve_epochs = 0
    trial_log_row = None
    prune_exception = None

    for epoch_idx in tqdm(range(1, int(epochs) + 1), leave=False, desc=f"{stage_label} | Trial {trial.number}"):
        model.train()
        for *features, batch_targets in train_loader:
            logits = model(*features)
            loss = loss_fn(logits, batch_targets)
            optimizer.zero_grad()
            loss.backward()
            with torch.no_grad():
                optimizer.step()

        valid_probs = predict_probabilities(valid_loader)
        valid_target_np = y_valid.cpu().numpy().ravel()
        valid_auc = safe_binary_metric(roc_auc_score, valid_target_np, valid_probs)
        valid_prauc = safe_binary_metric(average_precision_score, valid_target_np, valid_probs)
        valid_recall, valid_threshold, valid_fpr = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)

        trial_log_row = {
            "stage": stage_label,
            "trial": trial.number,
            "epoch": epoch_idx,
            "d_token": ft_params["d_token"],
            "n_blocks": ft_params["n_blocks"],
            "n_heads": ft_params["n_heads"],
            "ffn_hidden": ft_params["ffn_hidden"],
            "dropout": ft_params["dropout"],
            "lr": learning_rate,
            "batch": batch_size,
            "val_auc": valid_auc,
            "val_prauc": valid_prauc,
            "val_recall_at_fpr": valid_recall,
            "val_fpr": valid_fpr,
            "val_threshold": valid_threshold,
        }
        append_csv_row(TRIALS_CSV, TRIAL_LOG_FIELDS, trial_log_row)

        if valid_recall > best_valid_recall + EPSILON:
            best_valid_recall = valid_recall
            best_trial_row = trial_log_row
            best_model_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            best_validation_threshold = float(valid_threshold)
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        trial.report(best_valid_recall, epoch_idx)
        if trial.should_prune():
            prune_exception = optuna.TrialPruned(f"{stage_label} pruned at epoch {epoch_idx}")
            break

        if EARLY_STOP_PATIENCE > 0 and no_improve_epochs >= EARLY_STOP_PATIENCE:
            break

    if best_trial_row is None:
        if trial_log_row is None:
            return 0.0
        best_trial_row = trial_log_row

    append_csv_row(BEST_CSV, TRIAL_LOG_FIELDS, best_trial_row)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    valid_probs = predict_probabilities(valid_loader)
    recall_valid, _, fpr_valid = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)

    threshold_star = best_validation_threshold

    fairness_ratio_valid, fairness_attr_scores_valid, valid_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
        y_valid.cpu().numpy().ravel(), valid_probs, valid_sensitive_attributes_df, threshold_star
    )
    valid_attr_n_groups, valid_attr_is_degenerate = aequitas_attribute_metadata_by_attribute(valid_aequitas_disparity_df)
    fairness_age_valid = fairness_attr_scores_valid.get("age_group", float("nan"))
    fairness_income_valid = fairness_attr_scores_valid.get("income_group", float("nan"))
    fairness_employment_valid = fairness_attr_scores_valid.get("employment_status_group", float("nan"))
    aeq_attribute_n_distinct_groups_age_val = valid_attr_n_groups.get("age_group", float("nan"))
    aeq_attribute_n_distinct_groups_income_val = valid_attr_n_groups.get("income_group", float("nan"))
    aeq_attribute_n_distinct_groups_employment_status_val = valid_attr_n_groups.get("employment_status_group", float("nan"))
    aeq_attribute_is_degenerate_age_val = valid_attr_is_degenerate.get("age_group", float("nan"))
    aeq_attribute_is_degenerate_income_val = valid_attr_is_degenerate.get("income_group", float("nan"))
    aeq_attribute_is_degenerate_employment_status_val = valid_attr_is_degenerate.get("employment_status_group", float("nan"))

    if LOG_TEST_DURING_TUNING:
        test_probs = predict_probabilities(test_loader)
        y_test_np = y_test.cpu().numpy().ravel()
        y_test_pred = (test_probs >= threshold_star).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test_np, y_test_pred, labels=[0, 1]).ravel()
        recall_test = tp / (tp + fn + EPSILON)
        fpr_test = fp / (fp + tn + EPSILON)
        fairness_ratio_test, fairness_attr_scores_test, test_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
            y_test_np, test_probs, test_sensitive_attributes_df, threshold_star
        )
        test_attr_n_groups, test_attr_is_degenerate = aequitas_attribute_metadata_by_attribute(test_aequitas_disparity_df)
        fairness_age_test = fairness_attr_scores_test.get("age_group", float("nan"))
        fairness_income_test = fairness_attr_scores_test.get("income_group", float("nan"))
        fairness_employment_test = fairness_attr_scores_test.get("employment_status_group", float("nan"))
        aeq_attribute_n_distinct_groups_age_test = test_attr_n_groups.get("age_group", float("nan"))
        aeq_attribute_n_distinct_groups_income_test = test_attr_n_groups.get("income_group", float("nan"))
        aeq_attribute_n_distinct_groups_employment_status_test = test_attr_n_groups.get("employment_status_group", float("nan"))
        aeq_attribute_is_degenerate_age_test = test_attr_is_degenerate.get("age_group", float("nan"))
        aeq_attribute_is_degenerate_income_test = test_attr_is_degenerate.get("income_group", float("nan"))
        aeq_attribute_is_degenerate_employment_status_test = test_attr_is_degenerate.get("employment_status_group", float("nan"))
    else:
        recall_test = float("nan")
        fpr_test = float("nan")
        fairness_ratio_test = float("nan")
        fairness_age_test = float("nan")
        fairness_income_test = float("nan")
        fairness_employment_test = float("nan")
        test_attr_n_groups = {}
        test_attr_is_degenerate = {}
        aeq_attribute_n_distinct_groups_age_test = float("nan")
        aeq_attribute_n_distinct_groups_income_test = float("nan")
        aeq_attribute_n_distinct_groups_employment_status_test = float("nan")
        aeq_attribute_is_degenerate_age_test = float("nan")
        aeq_attribute_is_degenerate_income_test = float("nan")
        aeq_attribute_is_degenerate_employment_status_test = float("nan")

    fairness_attribute_n_distinct_groups_payload = json.dumps(
        {
            "val": {k: int(v) for k, v in valid_attr_n_groups.items()},
            "test": {k: int(v) for k, v in test_attr_n_groups.items()},
        },
        sort_keys=True,
    )
    fairness_attribute_is_degenerate_payload = json.dumps(
        {
            "val": {k: int(v) for k, v in valid_attr_is_degenerate.items()},
            "test": {k: int(v) for k, v in test_attr_is_degenerate.items()},
        },
        sort_keys=True,
    )

    append_csv_row(
        FAIR_CSV,
        FAIR_LOG_FIELDS,
        {
            "stage": stage_label,
            "trial": trial.number,
            "perf_recall_test": recall_test,
            "fairness_ratio_test": fairness_ratio_test,
            "thr_star": threshold_star,
            "fpr_test": fpr_test,
            "recall_val": recall_valid,
            "fpr_val": fpr_valid,
            "fairness_ratio_val": fairness_ratio_valid,
            "aeq_fpr_parity_age_test": fairness_age_test,
            "aeq_fpr_parity_income_test": fairness_income_test,
            "aeq_fpr_parity_employment_status_test": fairness_employment_test,
            "aeq_fpr_parity_age_val": fairness_age_valid,
            "aeq_fpr_parity_income_val": fairness_income_valid,
            "aeq_fpr_parity_employment_status_val": fairness_employment_valid,
            "attribute_n_distinct_groups": fairness_attribute_n_distinct_groups_payload,
            "attribute_is_degenerate": fairness_attribute_is_degenerate_payload,
            "aeq_attribute_n_distinct_groups_age_test": aeq_attribute_n_distinct_groups_age_test,
            "aeq_attribute_n_distinct_groups_income_test": aeq_attribute_n_distinct_groups_income_test,
            "aeq_attribute_n_distinct_groups_employment_status_test": aeq_attribute_n_distinct_groups_employment_status_test,
            "aeq_attribute_n_distinct_groups_age_val": aeq_attribute_n_distinct_groups_age_val,
            "aeq_attribute_n_distinct_groups_income_val": aeq_attribute_n_distinct_groups_income_val,
            "aeq_attribute_n_distinct_groups_employment_status_val": aeq_attribute_n_distinct_groups_employment_status_val,
            "aeq_attribute_is_degenerate_age_test": aeq_attribute_is_degenerate_age_test,
            "aeq_attribute_is_degenerate_income_test": aeq_attribute_is_degenerate_income_test,
            "aeq_attribute_is_degenerate_employment_status_test": aeq_attribute_is_degenerate_employment_status_test,
            "aeq_attribute_is_degenerate_age_val": aeq_attribute_is_degenerate_age_val,
            "aeq_attribute_is_degenerate_income_val": aeq_attribute_is_degenerate_income_val,
            "aeq_attribute_is_degenerate_employment_status_val": aeq_attribute_is_degenerate_employment_status_val,
        },
    )

    if prune_exception is not None:
        raise prune_exception

    return float(best_valid_recall)


## Block 26: FT-Transformer Hyperparameter Study

Runs one Optuna study over FT-Transformer backbone size, dropout, learning rate, and batch size.


In [ ]:
# ---------------- Run FT-Transformer tuning study ----------------
def create_tuning_study():
    return optuna.create_study(
        direction=OPTUNA_DIRECTION,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE_URL,
        load_if_exists=OPTUNA_LOAD_IF_EXISTS,
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=PRUNER_STARTUP_TRIALS,
            n_warmup_steps=PRUNER_WARMUP_STEPS,
        ),
    )


def optimize_to_target(study: optuna.Study, objective_fn, target_trials: int, study_label: str):
    existing_trials = len(study.trials)
    remaining_trials = max(0, int(target_trials) - int(existing_trials))
    print(
        f"[{study_label}] existing trials: {existing_trials} | target: {int(target_trials)} | remaining: {remaining_trials}"
    )

    if remaining_trials > 0:
        study.optimize(
            objective_fn,
            n_trials=remaining_trials,
            show_progress_bar=PROGRESS_BAR_ENABLED,
        )
    else:
        print(f"[{study_label}] target already reached; skipping new trials.")

    return remaining_trials


def objective_tuning(trial: optuna.Trial):
    ft_params = {
        "d_token": trial.suggest_categorical("d_token", HP_D_TOKEN_CHOICES),
        "n_blocks": trial.suggest_int("n_blocks", HP_N_BLOCKS_MIN, HP_N_BLOCKS_MAX),
        "n_heads": trial.suggest_categorical("n_heads", HP_N_HEADS_CHOICES),
        "ffn_hidden": trial.suggest_categorical("ffn_hidden", HP_FFN_HIDDEN_CHOICES),
        "dropout": trial.suggest_float("dropout", HP_DROPOUT_MIN, HP_DROPOUT_MAX),
    }
    learning_rate = trial.suggest_float("lr", HP_LR_MIN, HP_LR_MAX, log=True)
    batch_size = trial.suggest_categorical("batch", HP_BATCH_CHOICES)
    return run_single_trial(
        trial,
        stage_label="ft_dense",
        ft_params=ft_params,
        learning_rate=learning_rate,
        batch_size=batch_size,
        epochs=TUNING_EPOCHS,
    )


# Optuna progress bar can fail if ipywidgets is unavailable; disable safely
PROGRESS_BAR_ENABLED = False
try:
    import ipywidgets as _ipywidgets  # noqa: F401
    PROGRESS_BAR_ENABLED = True
except Exception:
    PROGRESS_BAR_ENABLED = False


if TUNING_TRIALS <= 0:
    raise ValueError("TUNING_TRIALS must be > 0 to run the FT-Transformer baseline study.")

print("Running FT-Transformer dense-head tuning study...")
tuning_study = create_tuning_study()
tuning_new_trials = optimize_to_target(tuning_study, objective_tuning, TUNING_TRIALS, "FT-Dense tuning")
final_best_stage = "ft_dense"
best_params = {
    "d_token": int(tuning_study.best_params["d_token"]),
    "n_blocks": int(tuning_study.best_params["n_blocks"]),
    "n_heads": int(tuning_study.best_params["n_heads"]),
    "ffn_hidden": int(tuning_study.best_params["ffn_hidden"]),
    "dropout": float(tuning_study.best_params["dropout"]),
    "lr": float(tuning_study.best_params["lr"]),
    "batch": int(tuning_study.best_params["batch"]),
}
print("[FT-Dense tuning] best recall@FPR cap:", tuning_study.best_value)
print("[FT-Dense tuning] best params:", tuning_study.best_params)

print("\nSelected final params (for retrain):")
print(best_params)
print("Selected label:", final_best_stage)


## Block 27: Retrain Initialization

Builds the final FT-Transformer dense baseline using the selected best parameters and prepares optimizer/loss/loaders.


In [ ]:
# ---------------- Retrain best config ----------------
model = FTTransformer(
    num_numeric_features,
    cat_cardinalities,
    d_token=best_params["d_token"],
    n_blocks=best_params["n_blocks"],
    n_heads=best_params["n_heads"],
    ffn_hidden=best_params["ffn_hidden"],
    dropout=best_params["dropout"],
).to(active_device)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCEWithLogitsLoss()

train_loader = build_loader(x_num_train, x_cat_train, y_train, best_params["batch"], shuffle=True)
valid_loader = build_loader(
    x_num_valid,
    x_cat_valid,
    y_valid,
    best_params["batch"] * VALID_TEST_BATCH_MULTIPLIER,
    shuffle=False,
)
test_loader = build_loader(
    x_num_test,
    x_cat_test,
    y_test,
    best_params["batch"] * VALID_TEST_BATCH_MULTIPLIER,
    shuffle=False,
)


## Block 28: Retrain Prediction Helper

Defines probability inference helper used during retraining and final evaluation.


In [ ]:
def predict_probabilities(data_loader):
    model.eval()
    probability_chunks = []
    with torch.no_grad():
        for *features, _ in data_loader:
            probability_chunks.append(torch.sigmoid(model(*features)).squeeze(1).cpu())
    return torch.cat(probability_chunks).numpy()


## Block 29: Retraining Loop

Retrains with best config, tracks best validation recall under FPR cap, and stores best model state.


In [ ]:
best_valid_recall = -1.0
best_model_state = None
best_epoch_idx = None
best_epoch_threshold = DEFAULT_THRESHOLD
retrain_history_rows = []

for epoch_idx in range(1, FINAL_EPOCHS + 1):
    epoch_start_time = time.time()
    model.train()
    epoch_loss_sum = 0.0
    epoch_sample_count = 0

    for *features, batch_targets in train_loader:
        logits = model(*features)
        loss = loss_fn(logits, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        with torch.no_grad():
            optimizer.step()

        batch_size_now = int(batch_targets.shape[0])
        epoch_loss_sum += float(loss.item()) * batch_size_now
        epoch_sample_count += batch_size_now

    valid_probs = predict_probabilities(valid_loader)
    valid_y_np = y_valid.cpu().numpy().ravel()
    valid_auc = safe_binary_metric(roc_auc_score, valid_y_np, valid_probs)
    valid_prauc = safe_binary_metric(average_precision_score, valid_y_np, valid_probs)
    valid_recall, threshold_star, valid_fpr = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)

    epoch_train_loss = epoch_loss_sum / max(epoch_sample_count, 1)
    epoch_seconds = float(time.time() - epoch_start_time)
    is_best_epoch = bool(valid_recall > best_valid_recall + EPSILON)

    if is_best_epoch:
        best_valid_recall = float(valid_recall)
        best_epoch_idx = int(epoch_idx)
        best_epoch_threshold = float(threshold_star)
        best_model_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }

    retrain_history_rows.append({
        'epoch': int(epoch_idx),
        'train_loss': float(epoch_train_loss),
        'val_auc': float(valid_auc) if np.isfinite(valid_auc) else float('nan'),
        'val_prauc': float(valid_prauc) if np.isfinite(valid_prauc) else float('nan'),
        'val_recall_at_fpr': float(valid_recall),
        'val_fpr': float(valid_fpr),
        'val_threshold': float(threshold_star),
        'is_best_epoch': int(is_best_epoch),
        'epoch_seconds': float(epoch_seconds),
    })

    if epoch_idx % 2 == 0 or is_best_epoch or epoch_idx == 1 or epoch_idx == FINAL_EPOCHS:
        print(
            f"[Retrain] Epoch {epoch_idx:02d} | loss={epoch_train_loss:.4f} | "
            f"VAL AUC={valid_auc:.4f} | VAL PR-AUC={valid_prauc:.4f} | "
            f"VAL recall@FPR<={int(FPR_CAP * 100)}%={valid_recall:.4f} | FPR={valid_fpr:.4f}" +
            (" | BEST" if is_best_epoch else "")
        )

if best_model_state is None:
    raise RuntimeError('Final retrain did not produce a best model checkpoint.')

model.load_state_dict(best_model_state)
retrain_history_df = pd.DataFrame(retrain_history_rows)
print(f"Loaded best retrain checkpoint from epoch {best_epoch_idx} (VAL recall@FPR cap={best_valid_recall:.4f}, thr={best_epoch_threshold:.6f})")

if not retrain_history_df.empty:
    display(retrain_history_df.tail(min(10, len(retrain_history_df))))



## Block 30: Final Evaluation

Computes validation and test performance metrics using selected threshold and prints final results.


In [ ]:
# ---------------- Final evaluation ----------------
from sklearn.metrics import auc as sklearn_auc, precision_recall_curve as sklearn_precision_recall_curve
valid_probs = predict_probabilities(valid_loader)
test_probs = predict_probabilities(test_loader)
print("\nVALID AUC:", roc_auc_score(y_valid.cpu().numpy(), valid_probs))
valid_pr_auc_ap = float(average_precision_score(y_valid.cpu().numpy(), valid_probs))
print("VALID PR-AUC (AP):", valid_pr_auc_ap)
valid_recall, threshold_star, valid_fpr = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)
print(f"VALID recall@FPR<={int(FPR_CAP * 100)}%: {valid_recall:.4f} | chosen_thr: {threshold_star:.6f} | FPR: {valid_fpr:.4f}")

valid_y_np = y_valid.cpu().numpy().ravel()
valid_pr_curve_precision, valid_pr_curve_recall, _ = sklearn_precision_recall_curve(valid_y_np, valid_probs)
valid_pr_auc_trapezoidal = float(sklearn_auc(valid_pr_curve_recall[::-1], valid_pr_curve_precision[::-1]))
print("VALID PR-AUC (trapezoidal full curve):", valid_pr_auc_trapezoidal)
y_valid_pred = (valid_probs >= threshold_star).astype(int)
valid_tn, valid_fp, valid_fn, valid_tp = confusion_matrix(valid_y_np, y_valid_pred, labels=[0, 1]).ravel()
valid_precision_at_selected_threshold = valid_tp / (valid_tp + valid_fp + EPSILON)
valid_dylan_snippet_pr_auc = float(sklearn_auc(
    [0.0, float(valid_recall), 1.0],
    [0.0, float(valid_precision_at_selected_threshold), 1.0],
))
print(
    "VALID Dylan snippet PR-AUC proxy (single threshold):",
    valid_dylan_snippet_pr_auc,
)

y_test_np = y_test.cpu().numpy().ravel()
test_pr_auc_ap = float(average_precision_score(y_test.cpu().numpy(), test_probs))
test_pr_curve_precision, test_pr_curve_recall, _ = sklearn_precision_recall_curve(y_test_np, test_probs)
test_pr_auc_trapezoidal = float(sklearn_auc(test_pr_curve_recall[::-1], test_pr_curve_precision[::-1]))
y_test_pred = (test_probs >= threshold_star).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_np, y_test_pred, labels=[0, 1]).ravel()
recall_test = tp / (tp + fn + EPSILON)
fpr_test = fp / (fp + tn + EPSILON)
precision_test_at_selected_threshold = tp / (tp + fp + EPSILON)
test_dylan_snippet_pr_auc = float(sklearn_auc(
    [0.0, float(recall_test), 1.0],
    [0.0, float(precision_test_at_selected_threshold), 1.0],
))
print(f"TEST  recall: {recall_test:.4f} | TEST FPR: {fpr_test:.4f}")
print(
    "TEST  Dylan snippet PR-AUC proxy (single threshold):",
    test_dylan_snippet_pr_auc,
)
print("TEST  AUC:", roc_auc_score(y_test.cpu().numpy(), test_probs))
print("TEST  PR-AUC (AP):", test_pr_auc_ap)
print("TEST  PR-AUC (trapezoidal full curve):", test_pr_auc_trapezoidal)
valid_fairness_overall, valid_fairness_attr_scores, valid_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
    y_valid.cpu().numpy().ravel(), valid_probs, valid_sensitive_attributes_df, threshold_star
)
test_fairness_overall, test_fairness_attr_scores, test_aequitas_disparity_df = aequitas_fpr_parity_summary_at_threshold(
    y_test_np, test_probs, test_sensitive_attributes_df, threshold_star
)

print(f"VALID Aequitas FPR parity (worst attribute/group, ideal=1): {valid_fairness_overall:.4f}")
print("VALID Aequitas FPR parity by attribute:", valid_fairness_attr_scores)
print(f"TEST  Aequitas FPR parity (worst attribute/group, ideal=1): {test_fairness_overall:.4f}")
print("TEST  Aequitas FPR parity by attribute:", test_fairness_attr_scores)

if not valid_aequitas_disparity_df.empty and "attribute_name" in valid_aequitas_disparity_df.columns:
    display_columns = [column_name for column_name in [
        "attribute_name", "attribute_value", "attribute_n_distinct_groups", "attribute_is_degenerate", "fpr", "fpr_disparity", "score", "label_value"
    ] if column_name in valid_aequitas_disparity_df.columns]
    tracked_attributes = set(AEQUITAS_FAIRNESS_ATTRIBUTES)
    display(valid_aequitas_disparity_df.loc[valid_aequitas_disparity_df["attribute_name"].isin(tracked_attributes), display_columns])

if not test_aequitas_disparity_df.empty and "attribute_name" in test_aequitas_disparity_df.columns:
    display_columns = [column_name for column_name in [
        "attribute_name", "attribute_value", "attribute_n_distinct_groups", "attribute_is_degenerate", "fpr", "fpr_disparity", "score", "label_value"
    ] if column_name in test_aequitas_disparity_df.columns]
    tracked_attributes = set(AEQUITAS_FAIRNESS_ATTRIBUTES)
    display(test_aequitas_disparity_df.loc[test_aequitas_disparity_df["attribute_name"].isin(tracked_attributes), display_columns])



## Block 31: Artifact Export

Saves trained model weights and metadata JSON, then prints output paths.


In [ ]:
# ---------------- Save artifacts ----------------
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), EXPORT_DIR / "model.pt")

final_metrics_summary = None
if all(name in globals() for name in [
    'valid_probs', 'test_probs', 'threshold_star', 'valid_recall', 'valid_fpr',
    'recall_test', 'fpr_test', 'valid_fairness_overall', 'test_fairness_overall'
]):
    final_metrics_summary = {
        'valid_auc': float(roc_auc_score(y_valid.cpu().numpy(), valid_probs)),
        'valid_prauc': float(average_precision_score(y_valid.cpu().numpy(), valid_probs)),
        'valid_recall_at_fpr_cap': float(valid_recall),
        'valid_fpr_at_selected_threshold': float(valid_fpr),
        'test_auc': float(roc_auc_score(y_test.cpu().numpy(), test_probs)),
        'test_prauc': float(average_precision_score(y_test.cpu().numpy(), test_probs)),
        'test_recall_at_selected_threshold': float(recall_test),
        'test_fpr_at_selected_threshold': float(fpr_test),
        'threshold_star': float(threshold_star),
        'valid_aequitas_fpr_parity_overall': float(valid_fairness_overall),
        'test_aequitas_fpr_parity_overall': float(test_fairness_overall),
    }

with open(EXPORT_DIR / "meta.json", "w") as meta_file:
    json.dump({
        "experiment_tag": EXPERIMENT_TAG,
        "dataset_csv": str(DATA_CSV_PATH),
        "target": target_col,
        "cat_cols": categorical_cols,
        "cont_cols": continuous_cols,
        "fpr_cap": FPR_CAP,
        "threshold": float(threshold_star),
        "selected_stage": final_best_stage,
        "optuna_study_name": OPTUNA_STUDY_NAME,
        "tuning_best_params": tuning_study.best_params,
        "retrain_best_epoch": int(best_epoch_idx) if 'best_epoch_idx' in globals() and best_epoch_idx is not None else None,
        "final_metrics_summary": final_metrics_summary,
        **best_params,
    }, meta_file, indent=2)

retrain_history_path = CSV_EXPORT_DIR / 'retrain_history_final_train.csv'
if 'retrain_history_df' in globals() and isinstance(retrain_history_df, pd.DataFrame) and not retrain_history_df.empty:
    retrain_history_df.to_csv(retrain_history_path, index=False)
    print(f"Retrain history CSV : {retrain_history_path}")
else:
    print('Retrain history CSV : not available (retrain_history_df missing/empty)')

print(f"\nSaved model + meta to {EXPORT_DIR}/")
print(f"Optuna DB         : {OPTUNA_STORAGE_PATH}")
print(f"CSV export dir    : {CSV_EXPORT_DIR}")
print(f"Per-epoch trial log: {TRIALS_CSV}")
print(f"Best-per-trial log : {BEST_CSV}")
print(f"Fairness log       : {FAIR_CSV}")


## Block 31A: Export CSV Sanity Checks

Validates exported trial/best/fairness CSV files for schema, NaNs, metric ranges, and cross-file consistency.


In [ ]:
# ---------------- Export CSV sanity checks ----------------
trials_df = pd.read_csv(TRIALS_CSV)
best_df = pd.read_csv(BEST_CSV)
fair_df = pd.read_csv(FAIR_CSV)

sanity_errors = []
sanity_warnings = []


def require_columns(df, required_cols, file_label):
    missing_cols = [column_name for column_name in required_cols if column_name not in df.columns]
    if missing_cols:
        sanity_errors.append(f"{file_label} missing columns: {missing_cols}")


def check_numeric_range(df, column_name, low, high, file_label, allow_nan=False):
    if column_name not in df.columns:
        return
    values = pd.to_numeric(df[column_name], errors="coerce")
    nan_count = int(values.isna().sum())
    if nan_count > 0 and not allow_nan:
        sanity_errors.append(f"{file_label}.{column_name} has {nan_count} NaN values.")
    finite_values = values[np.isfinite(values)]
    if len(finite_values) == 0:
        return
    out_of_range = int(((finite_values < low) | (finite_values > high)).sum())
    if out_of_range > 0:
        sanity_errors.append(
            f"{file_label}.{column_name} has {out_of_range} values outside [{low}, {high}]."
        )


require_columns(trials_df, TRIAL_LOG_FIELDS, "TRIALS_CSV")
require_columns(best_df, TRIAL_LOG_FIELDS, "BEST_CSV")
require_columns(fair_df, FAIR_LOG_FIELDS, "FAIR_CSV")

pair_columns = {"stage", "trial"}
if pair_columns.issubset(best_df.columns) and pair_columns.issubset(fair_df.columns) and pair_columns.issubset(trials_df.columns):
    if best_df.duplicated(["stage", "trial"]).any():
        sanity_errors.append("BEST_CSV has duplicate (stage, trial) rows.")
    if fair_df.duplicated(["stage", "trial"]).any():
        sanity_errors.append("FAIR_CSV has duplicate (stage, trial) rows.")

    best_pairs = set(map(tuple, best_df[["stage", "trial"]].to_records(index=False)))
    fair_pairs = set(map(tuple, fair_df[["stage", "trial"]].to_records(index=False)))
    trial_pairs = set(map(tuple, trials_df[["stage", "trial"]].drop_duplicates().to_records(index=False)))

    if best_pairs != fair_pairs:
        sanity_errors.append(
            f"BEST/FAIR mismatch: best_only={len(best_pairs - fair_pairs)}, fair_only={len(fair_pairs - best_pairs)}"
        )
    if not best_pairs.issubset(trial_pairs):
        sanity_errors.append("Some BEST_CSV trials are missing from TRIALS_CSV.")
else:
    sanity_errors.append("One or more CSVs are missing stage/trial columns required for consistency checks.")

# Core metric ranges.
for file_label, dataframe in [("TRIALS_CSV", trials_df), ("BEST_CSV", best_df)]:
    check_numeric_range(dataframe, "val_auc", 0.0, 1.0, file_label, allow_nan=True)
    check_numeric_range(dataframe, "val_prauc", 0.0, 1.0, file_label, allow_nan=True)
    check_numeric_range(dataframe, "val_recall_at_fpr", 0.0, 1.0, file_label)
    check_numeric_range(dataframe, "val_fpr", 0.0, 1.0, file_label)

check_numeric_range(fair_df, "recall_val", 0.0, 1.0, "FAIR_CSV")
check_numeric_range(fair_df, "fpr_val", 0.0, 1.0, "FAIR_CSV")
check_numeric_range(fair_df, "fairness_ratio_val", 0.0, 1.0, "FAIR_CSV")
for column_name in [
    "aeq_fpr_parity_age_val",
    "aeq_fpr_parity_income_val",
    "aeq_fpr_parity_employment_status_val",
]:
    check_numeric_range(fair_df, column_name, 0.0, 1.0, "FAIR_CSV", allow_nan=True)

if "val_fpr" in best_df.columns:
    cap_violations = int((pd.to_numeric(best_df["val_fpr"], errors="coerce") > (FPR_CAP + 1e-6)).sum())
    if cap_violations > 0:
        sanity_warnings.append(f"BEST_CSV has {cap_violations} rows with val_fpr > FPR_CAP ({FPR_CAP}).")

if LOG_TEST_DURING_TUNING:
    check_numeric_range(fair_df, "perf_recall_test", 0.0, 1.0, "FAIR_CSV")
    check_numeric_range(fair_df, "fpr_test", 0.0, 1.0, "FAIR_CSV")
    check_numeric_range(fair_df, "fairness_ratio_test", 0.0, 1.0, "FAIR_CSV")
    for column_name in [
        "aeq_fpr_parity_age_test",
        "aeq_fpr_parity_income_test",
        "aeq_fpr_parity_employment_status_test",
    ]:
        check_numeric_range(fair_df, column_name, 0.0, 1.0, "FAIR_CSV", allow_nan=True)
else:
    for test_column_name in [
        "perf_recall_test",
        "fpr_test",
        "fairness_ratio_test",
        "aeq_fpr_parity_age_test",
        "aeq_fpr_parity_income_test",
        "aeq_fpr_parity_employment_status_test",
    ]:
        if test_column_name in fair_df.columns and fair_df[test_column_name].notna().any():
            sanity_warnings.append(
                f"{test_column_name} has non-NaN values even though LOG_TEST_DURING_TUNING is False."
            )

if "fairness_ratio_val" in fair_df.columns:
    max_fairness_val = float(pd.to_numeric(fair_df["fairness_ratio_val"], errors="coerce").max())
    if max_fairness_val > 1.0 + 1e-6:
        sanity_warnings.append(
            "fairness_ratio_val exceeds 1.0, which suggests malformed fairness outputs (Aequitas parity summary should stay within [0, 1])."
        )

valid_positive_rate = float(y_valid.cpu().numpy().mean())
if "val_prauc" in best_df.columns:
    best_val_prauc_median = float(pd.to_numeric(best_df["val_prauc"], errors="coerce").median())
else:
    best_val_prauc_median = float("nan")

if np.isfinite(best_val_prauc_median):
    pr_auc_lift = best_val_prauc_median / max(valid_positive_rate, EPSILON)
    print(f"Sanity context | VALID positive rate: {valid_positive_rate:.6f} | median BEST val_prauc: {best_val_prauc_median:.6f} | lift: {pr_auc_lift:.2f}x")
else:
    sanity_warnings.append("BEST_CSV.val_prauc is missing or non-numeric; PR-AUC context was skipped.")

if sanity_warnings:
    print("CSV sanity warnings:")
    for warning_message in sanity_warnings:
        print(" -", warning_message)

if sanity_errors:
    print("CSV sanity check: FAILED")
    for error_message in sanity_errors:
        print(" -", error_message)
    raise ValueError("Export CSV sanity check failed. See messages above.")

print("CSV sanity check: PASSED")
print(f"Rows | TRIALS={len(trials_df)} BEST={len(best_df)} FAIR={len(fair_df)}")


## Block 32: Plotting Setup

Loads fairness trial log as dataframe for visualization.


In [ ]:
# ---------------- Plots (Aequitas fairness vs performance) ----------------
fairness_results_df = pd.read_csv(FAIR_CSV)

use_test_metrics_for_plot = (
    LOG_TEST_DURING_TUNING
    and "perf_recall_test" in fairness_results_df.columns
    and fairness_results_df["perf_recall_test"].notna().any()
    and fairness_results_df["fairness_ratio_test"].notna().any()
)

if use_test_metrics_for_plot:
    perf_col = "perf_recall_test"
    fair_col = "fairness_ratio_test"
    split_label = "TEST"
else:
    perf_col = "recall_val"
    fair_col = "fairness_ratio_val"
    split_label = "VALID"


## Block 33: Full Fairness-Performance Scatter

Generates and saves the full scatter plot of trial fairness vs performance.


In [ ]:
# 1) Full scatter of all trials.
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
ax.scatter(fairness_results_df[perf_col], fairness_results_df[fair_col], alpha=0.7)

top_trials_by_recall = fairness_results_df.sort_values(perf_col, ascending=False).head(TOP_K_TRIALS)
ax.scatter(
    top_trials_by_recall[perf_col],
    top_trials_by_recall[fair_col],
    s=70,
    edgecolors="black",
)
ax.set_xlabel(f"Performance (Recall @ {int(FPR_CAP * 100)}% FPR) - {split_label}")
ax.set_ylabel(f"Fairness (Aequitas FPR parity summary) - {split_label} (ideal = 1)")
ax.set_title("Aequitas Fairness vs Performance (all trials)")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "fig_fairness_vs_performance.png", bbox_inches="tight")
print("Saved:", EXPORT_DIR / "fig_fairness_vs_performance.png")


## Block 34: Zoomed Fairness-Performance Scatter

Generates and saves a zoomed-in scatter focused on high-performance region.


In [ ]:
# 2) Zoom into high-performance region.
zoom_xmin = max(
    fairness_results_df[perf_col].min(),
    fairness_results_df[perf_col].quantile(ZOOM_X_QUANTILE),
)
zoom_xmax = fairness_results_df[perf_col].max()
zoom_ymin = fairness_results_df[fair_col].quantile(ZOOM_Y_LOW_QUANTILE)
zoom_ymax = fairness_results_df[fair_col].quantile(ZOOM_Y_HIGH_QUANTILE)

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
ax.scatter(fairness_results_df[perf_col], fairness_results_df[fair_col], alpha=0.5)
ax.scatter(
    top_trials_by_recall[perf_col],
    top_trials_by_recall[fair_col],
    s=70,
    edgecolors="black",
)
ax.set_xlim(zoom_xmin, zoom_xmax + 1e-6)
ax.set_ylim(zoom_ymin - 1e-3, zoom_ymax + 1e-3)
ax.set_xlabel(f"Performance (Recall @ {int(FPR_CAP * 100)}% FPR) - {split_label}")
ax.set_ylabel(f"Fairness (Aequitas FPR parity summary) - {split_label} (ideal = 1)")
ax.set_title("Aequitas Fairness vs Performance (zoom)")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "fig_fairness_vs_performance_zoom.png", bbox_inches="tight")
print("Saved:", EXPORT_DIR / "fig_fairness_vs_performance_zoom.png")


## Block 35: Paper Artifact Helpers

Defines reusable helpers for exporting paper tables (CSV), curve points, and figure files from this advanced experiment run.


In [ ]:
# ---------------- Paper artifact helpers (tables + diagrams for ECML paper) ----------------
from sklearn.metrics import precision_recall_curve, auc
from sklearn.calibration import calibration_curve

try:
    import seaborn as sns
except Exception:
    sns = None

PAPER_ARTIFACT_DIR = PAPER_EXPORT_DIR
PAPER_TABLES_DIR = PAPER_ARTIFACT_DIR / 'tables'
PAPER_CURVES_DIR = PAPER_ARTIFACT_DIR / 'curves'
PAPER_FIGS_DIR = PAPER_ARTIFACT_DIR / 'figures'
PAPER_PREDS_DIR = PAPER_ARTIFACT_DIR / 'predictions'
for _dir in [PAPER_ARTIFACT_DIR, PAPER_TABLES_DIR, PAPER_CURVES_DIR, PAPER_FIGS_DIR, PAPER_PREDS_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

paper_artifact_registry = []


def register_artifact(path, kind, description):
    path = Path(path)
    paper_artifact_registry.append({
        'kind': str(kind),
        'path': str(path),
        'description': str(description),
        'exists': int(path.exists()),
    })
    return path


def save_df_csv(df: pd.DataFrame, path: Path, description: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    register_artifact(path, 'csv', description)
    return path


def save_fig(fig, path: Path, description: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    register_artifact(path, 'figure', description)
    return path


def safe_div(num, den):
    den = float(den)
    return float(num) / den if den != 0 else float('nan')


def dylan_snippet_pr_score_from_point(precision, recall):
    #single point threshold PR proxy
    try:
        precision = float(precision)
        recall = float(recall)
    except Exception:
        return float('nan')
    if not (np.isfinite(precision) and np.isfinite(recall)):
        return float('nan')
    precision = float(np.clip(precision, 0.0, 1.0))
    recall = float(np.clip(recall, 0.0, 1.0))
    return float(auc([0.0, recall, 1.0], [0.0, precision, 1.0]))


def trapezoidal_pr_auc_from_scores(y_true, y_prob):
    # full-curve PR area using trapezoidal integration over PR points
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(recall[::-1], precision[::-1]))


def confusion_metrics_from_probs(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    fpr = safe_div(fp, fp + tn)
    specificity = safe_div(tn, tn + fp)
    npv = safe_div(tn, tn + fn)
    f1 = safe_div(2 * precision * recall, precision + recall) if np.isfinite(precision) and np.isfinite(recall) else float('nan')
    acc = safe_div(tp + tn, tp + tn + fp + fn)
    bal_acc = np.nanmean([recall, specificity])
    return {
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'precision': float(precision) if np.isfinite(precision) else float('nan'),
        'recall': float(recall) if np.isfinite(recall) else float('nan'),
        'fpr': float(fpr) if np.isfinite(fpr) else float('nan'),
        'specificity': float(specificity) if np.isfinite(specificity) else float('nan'),
        'npv': float(npv) if np.isfinite(npv) else float('nan'),
        'f1': float(f1) if np.isfinite(f1) else float('nan'),
        'accuracy': float(acc) if np.isfinite(acc) else float('nan'),
        'balanced_accuracy': float(bal_acc) if np.isfinite(bal_acc) else float('nan'),
        'positive_rate_true': float(np.mean(y_true)),
        'positive_rate_pred': float(np.mean(y_pred)),
        'n_samples': int(y_true.shape[0]),
    }


def make_threshold_sweep_df(y_true, y_prob, selected_threshold, n_points=401):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    thresholds = np.linspace(0.0, 1.0, int(n_points))
    thresholds = np.unique(np.concatenate([thresholds, [float(selected_threshold)]]))
    rows = []
    for thr in thresholds:
        m = confusion_metrics_from_probs(y_true, y_prob, thr)
        rows.append({
            'threshold': float(thr),
            'recall': m['recall'],
            'fpr': m['fpr'],
            'precision': m['precision'],
            'specificity': m['specificity'],
            'f1': m['f1'],
            'positive_rate_pred': m['positive_rate_pred'],
            'within_fpr_cap': int(np.isfinite(m['fpr']) and m['fpr'] <= FPR_CAP + 1e-12),
            'is_selected_threshold': int(abs(float(thr) - float(selected_threshold)) < 1e-12),
        })
    sweep_df = pd.DataFrame(rows).sort_values('threshold').reset_index(drop=True)
    if not sweep_df.empty:
        under_cap = sweep_df[sweep_df['within_fpr_cap'] == 1]
        if not under_cap.empty and under_cap['recall'].notna().any():
            best_idx = under_cap['recall'].astype(float).idxmax()
            sweep_df['is_best_recall_under_cap'] = 0
            sweep_df.loc[best_idx, 'is_best_recall_under_cap'] = 1
        else:
            sweep_df['is_best_recall_under_cap'] = 0
    return sweep_df


def make_roc_df(y_true, y_prob):
    fpr, tpr, thr = roc_curve(np.asarray(y_true).astype(int).ravel(), np.asarray(y_prob, dtype=float).ravel())
    return pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thr})


def make_pr_df(y_true, y_prob):
    precision, recall, thr = precision_recall_curve(np.asarray(y_true).astype(int).ravel(), np.asarray(y_prob, dtype=float).ravel())
    threshold_series = np.full(shape=(len(precision),), fill_value=np.nan, dtype=float)
    threshold_series[:len(thr)] = thr
    return pd.DataFrame({'precision': precision, 'recall': recall, 'threshold': threshold_series})


def make_calibration_df(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    try:
        frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=int(n_bins), strategy='quantile')
        calib_df = pd.DataFrame({'mean_predicted_prob': mean_pred, 'fraction_positives': frac_pos})
    except Exception:
        calib_df = pd.DataFrame(columns=['mean_predicted_prob', 'fraction_positives'])
    calib_df['n_bins_requested'] = int(n_bins)
    return calib_df


def build_prediction_export_df(y_true, y_prob, threshold, sensitive_df, split_label):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)
    export_df = pd.DataFrame({
        'split': str(split_label),
        'row_idx': np.arange(len(y_true), dtype=int),
        'y_true': y_true,
        'y_prob': y_prob,
        'y_pred_at_selected_threshold': y_pred,
    })
    if isinstance(sensitive_df, pd.DataFrame):
        for col in ['age_group', 'income_group', 'employment_status_group']:
            if col in sensitive_df.columns and len(sensitive_df) == len(export_df):
                export_df[col] = sensitive_df[col].astype(str).to_numpy()
    return export_df


def build_subgroup_metrics_df(y_true, y_prob, threshold, sensitive_df, split_label):
    rows = []
    if not isinstance(sensitive_df, pd.DataFrame) or sensitive_df.empty:
        return pd.DataFrame(rows)

    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= float(threshold)).astype(int)

    for attr in [a for a in AEQUITAS_FAIRNESS_ATTRIBUTES if a in sensitive_df.columns]:
        values = sensitive_df[attr].astype(str).fillna('Unknown')
        for group_value in sorted(values.unique().tolist()):
            mask = (values == group_value).to_numpy()
            if int(mask.sum()) == 0:
                continue
            yt = y_true[mask]
            yp = y_prob[mask]
            yhat = y_pred[mask]
            tn, fp, fn, tp = confusion_matrix(yt, yhat, labels=[0, 1]).ravel()
            rows.append({
                'split': str(split_label),
                'attribute_name': str(attr),
                'attribute_value': str(group_value),
                'n_samples': int(mask.sum()),
                'positive_rate_true': float(np.mean(yt)),
                'positive_rate_pred': float(np.mean(yhat)),
                'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
                'precision': safe_div(tp, tp + fp),
                'recall': safe_div(tp, tp + fn),
                'fpr': safe_div(fp, fp + tn),
                'specificity': safe_div(tn, tn + fp),
                'f1': safe_div(2 * safe_div(tp, tp + fp) * safe_div(tp, tp + fn), safe_div(tp, tp + fp) + safe_div(tp, tp + fn)),
                'auc': safe_binary_metric(roc_auc_score, yt, yp),
                'pr_auc': safe_binary_metric(average_precision_score, yt, yp),
            })
    return pd.DataFrame(rows)


def summarize_split_frame(df: pd.DataFrame, split_label: str):
    if df is None or len(df) == 0:
        return {'split': split_label, 'n_samples': 0}
    month_min = int(df[MONTH_COLUMN].min()) if MONTH_COLUMN in df.columns else None
    month_max = int(df[MONTH_COLUMN].max()) if MONTH_COLUMN in df.columns else None
    target_rate = float(df[target_col].mean()) if target_col in df.columns else float('nan')
    return {
        'split': str(split_label),
        'n_samples': int(len(df)),
        'n_features_total': int(len(feature_cols)),
        'n_categorical_features': int(len(categorical_cols)),
        'n_continuous_features': int(len(continuous_cols)),
        'positive_rate': target_rate,
        'positive_count': int(df[target_col].sum()) if target_col in df.columns else None,
        'negative_count': int((1 - df[target_col]).sum()) if target_col in df.columns else None,
        'month_min': month_min,
        'month_max': month_max,
    }


def prepare_aequitas_export_df(disparity_df: pd.DataFrame, split_label: str):
    if disparity_df is None or len(disparity_df) == 0:
        return pd.DataFrame()
    out = disparity_df.copy()
    out.insert(0, 'split', str(split_label))
    if 'attribute_name' in out.columns:
        out = out[out['attribute_name'].isin(AEQUITAS_FAIRNESS_ATTRIBUTES)].copy()
    preferred_cols = [
        'split', 'attribute_name', 'attribute_value', 'attribute_n_distinct_groups', 'attribute_is_degenerate', 'label_value', 'score',
        'fpr', 'fpr_disparity', 'tpr', 'tpr_disparity', 'tnr', 'tnr_disparity',
        'fnr', 'fnr_disparity', 'pprev', 'ppr', 'ppr_disparity'
    ]
    keep = [c for c in preferred_cols if c in out.columns]
    extras = [c for c in out.columns if c not in keep]
    return out[keep + extras]


def build_attr_parity_summary_df(valid_scores: dict, test_scores: dict):
    attrs = sorted(set(list(valid_scores.keys()) + list(test_scores.keys())))
    rows = []
    for attr in attrs:
        rows.append({
            'attribute_name': attr,
            'valid_fpr_parity_score': float(valid_scores.get(attr, np.nan)),
            'test_fpr_parity_score': float(test_scores.get(attr, np.nan)),
        })
    return pd.DataFrame(rows)

print('Paper artifact directories ready:')
print(' -', PAPER_ARTIFACT_DIR)
print(' -', PAPER_TABLES_DIR)
print(' -', PAPER_CURVES_DIR)
print(' -', PAPER_FIGS_DIR)
print(' -', PAPER_PREDS_DIR)





## Block 36: Paper Tables and CSV Exports

Exports paper-ready CSVs for final metrics, split summaries, curve points, subgroup fairness tables, Aequitas outputs, tuning summaries, and prediction outputs from this advanced experiment.


In [ ]:
# ---------------- Paper tables / CSV exports ----------------
# Ensure CSV logs are loaded (cells above usually already created them).
if 'trials_df' not in globals():
    trials_df = pd.read_csv(TRIALS_CSV)
if 'best_df' not in globals():
    best_df = pd.read_csv(BEST_CSV)
if 'fair_df' not in globals():
    fair_df = pd.read_csv(FAIR_CSV)

valid_y_np = y_valid.cpu().numpy().ravel()
test_y_np = y_test.cpu().numpy().ravel()

valid_metrics_at_thr = confusion_metrics_from_probs(valid_y_np, valid_probs, threshold_star)
test_metrics_at_thr = confusion_metrics_from_probs(test_y_np, test_probs, threshold_star)

valid_auc = float(roc_auc_score(valid_y_np, valid_probs))
valid_prauc = float(average_precision_score(valid_y_np, valid_probs))
valid_pr_auc_trapezoidal = float(trapezoidal_pr_auc_from_scores(valid_y_np, valid_probs))
test_auc = float(roc_auc_score(test_y_np, test_probs))
test_prauc = float(average_precision_score(test_y_np, test_probs))
test_pr_auc_trapezoidal = float(trapezoidal_pr_auc_from_scores(test_y_np, test_probs))

final_metrics_df = pd.DataFrame([
    {
        'split': 'VALID',
        'auc': valid_auc,
        'pr_auc': valid_prauc,
        'pr_average_precision_score': valid_prauc,
        'pr_auc_trapezoidal': valid_pr_auc_trapezoidal,
        'pr_dylan_snippet_proxy': dylan_snippet_pr_score_from_point(valid_metrics_at_thr['precision'], valid_metrics_at_thr['recall']),
        'recall_at_selected_threshold': valid_metrics_at_thr['recall'],
        'fpr_at_selected_threshold': valid_metrics_at_thr['fpr'],
        'precision_at_selected_threshold': valid_metrics_at_thr['precision'],
        'f1_at_selected_threshold': valid_metrics_at_thr['f1'],
        'selected_threshold': float(threshold_star),
        'fpr_cap': float(FPR_CAP),
        'aequitas_fpr_parity_overall': float(valid_fairness_overall),
        'positive_rate_true': valid_metrics_at_thr['positive_rate_true'],
        'positive_rate_pred': valid_metrics_at_thr['positive_rate_pred'],
        'n_samples': valid_metrics_at_thr['n_samples'],
    },
    {
        'split': 'TEST',
        'auc': test_auc,
        'pr_auc': test_prauc,
        'pr_average_precision_score': test_prauc,
        'pr_auc_trapezoidal': test_pr_auc_trapezoidal,
        'pr_dylan_snippet_proxy': dylan_snippet_pr_score_from_point(test_metrics_at_thr['precision'], test_metrics_at_thr['recall']),
        'recall_at_selected_threshold': test_metrics_at_thr['recall'],
        'fpr_at_selected_threshold': test_metrics_at_thr['fpr'],
        'precision_at_selected_threshold': test_metrics_at_thr['precision'],
        'f1_at_selected_threshold': test_metrics_at_thr['f1'],
        'selected_threshold': float(threshold_star),
        'fpr_cap': float(FPR_CAP),
        'aequitas_fpr_parity_overall': float(test_fairness_overall),
        'positive_rate_true': test_metrics_at_thr['positive_rate_true'],
        'positive_rate_pred': test_metrics_at_thr['positive_rate_pred'],
        'n_samples': test_metrics_at_thr['n_samples'],
    },
])
save_df_csv(final_metrics_df, PAPER_TABLES_DIR / 'paper_table_final_metrics.csv', 'Final VALID/TEST performance + fairness summary at selected threshold')

confusion_summary_df = pd.DataFrame([
    {'split': 'VALID', 'threshold': float(threshold_star), **valid_metrics_at_thr},
    {'split': 'TEST', 'threshold': float(threshold_star), **test_metrics_at_thr},
])
save_df_csv(confusion_summary_df, PAPER_TABLES_DIR / 'paper_table_confusion_summary.csv', 'Confusion matrix counts and derived metrics at selected threshold')

pr_metric_comparison_df = pd.DataFrame([
    {
        'split': 'VALID',
        'selected_threshold': float(threshold_star),
        'precision_at_selected_threshold': float(valid_metrics_at_thr['precision']) if np.isfinite(valid_metrics_at_thr['precision']) else float('nan'),
        'recall_at_selected_threshold': float(valid_metrics_at_thr['recall']) if np.isfinite(valid_metrics_at_thr['recall']) else float('nan'),
        'pr_average_precision_score': float(average_precision_score(valid_y_np, valid_probs)),
        'pr_auc_trapezoidal': float(trapezoidal_pr_auc_from_scores(valid_y_np, valid_probs)),
        'pr_dylan_snippet_auc': dylan_snippet_pr_score_from_point(valid_metrics_at_thr['precision'], valid_metrics_at_thr['recall']),
    },
    {
        'split': 'TEST',
        'selected_threshold': float(threshold_star),
        'precision_at_selected_threshold': float(test_metrics_at_thr['precision']) if np.isfinite(test_metrics_at_thr['precision']) else float('nan'),
        'recall_at_selected_threshold': float(test_metrics_at_thr['recall']) if np.isfinite(test_metrics_at_thr['recall']) else float('nan'),
        'pr_average_precision_score': float(average_precision_score(test_y_np, test_probs)),
        'pr_auc_trapezoidal': float(trapezoidal_pr_auc_from_scores(test_y_np, test_probs)),
        'pr_dylan_snippet_auc': dylan_snippet_pr_score_from_point(test_metrics_at_thr['precision'], test_metrics_at_thr['recall']),
    },
])
save_df_csv(pr_metric_comparison_df, PAPER_TABLES_DIR / 'paper_table_pr_metric_comparison.csv', 'PR summary comparison: average_precision_score, trapezoidal PR-AUC, and Dylan snippet single-point proxy')

split_summary_df = pd.DataFrame([
    summarize_split_frame(train_df, 'TRAIN'),
    summarize_split_frame(valid_df, 'VALID'),
    summarize_split_frame(test_df, 'TEST'),
])
save_df_csv(split_summary_df, PAPER_TABLES_DIR / 'paper_table_dataset_split_summary.csv', 'Dataset split summary (counts, prevalence, month ranges)')

feature_summary_df = pd.DataFrame([
    {'table': 'summary', 'n_total_features': len(feature_cols), 'n_categorical_features': len(categorical_cols), 'n_continuous_features': len(continuous_cols)}
])
save_df_csv(feature_summary_df, PAPER_TABLES_DIR / 'paper_table_feature_summary.csv', 'Feature-type summary counts')

feature_inventory_df = pd.DataFrame({
    'feature_name': feature_cols,
    'inferred_type': ['categorical' if c in categorical_cols else 'continuous' for c in feature_cols],
    'dtype': [str(full_df[c].dtype) for c in feature_cols],
    'n_unique': [int(full_df[c].nunique(dropna=False)) for c in feature_cols],
})
save_df_csv(feature_inventory_df, PAPER_TABLES_DIR / 'paper_table_feature_inventory.csv', 'Feature inventory for appendix/supplement')

best_params_export_df = pd.DataFrame([
    {
        'experiment_tag': EXPERIMENT_TAG,
        'dataset_csv': str(DATA_CSV_PATH),
        'selected_stage': final_best_stage,
        'selected_threshold': float(threshold_star),
        'fpr_cap': float(FPR_CAP),
        **{k: v for k, v in best_params.items()}
    }
])
save_df_csv(best_params_export_df, PAPER_TABLES_DIR / 'paper_table_selected_params.csv', 'Selected hyperparameters and operating threshold')

if 'retrain_history_df' in globals() and isinstance(retrain_history_df, pd.DataFrame) and not retrain_history_df.empty:
    save_df_csv(retrain_history_df.copy(), PAPER_TABLES_DIR / 'paper_table_retrain_history.csv', 'Per-epoch final retrain history (loss + validation metrics)')
    retrain_best_row_df = retrain_history_df.loc[retrain_history_df['is_best_epoch'] == 1].copy()
    if not retrain_best_row_df.empty:
        save_df_csv(retrain_best_row_df, PAPER_TABLES_DIR / 'paper_table_retrain_best_epoch.csv', 'Best epoch row from final retrain history')

# Tuning summaries.
best_metrics_for_merge = best_df.copy()
fair_metrics_for_merge = fair_df.copy()
merged_trial_summary_df = best_metrics_for_merge.merge(
    fair_metrics_for_merge,
    on=['stage', 'trial'],
    how='left',
    suffixes=('_best', '_fair')
)
save_df_csv(merged_trial_summary_df, PAPER_TABLES_DIR / 'paper_table_trials_best_plus_fairness.csv', 'Merged best-per-trial metrics with fairness summary')

stage_summary_rows = []
for stage_name, stage_best in best_df.groupby('stage'):
    stage_rows = merged_trial_summary_df[merged_trial_summary_df['stage'] == stage_name]
    stage_summary_rows.append({
        'stage': stage_name,
        'n_trials_best_rows': int(len(stage_best)),
        'val_recall_at_fpr_mean': float(pd.to_numeric(stage_best['val_recall_at_fpr'], errors='coerce').mean()),
        'val_recall_at_fpr_max': float(pd.to_numeric(stage_best['val_recall_at_fpr'], errors='coerce').max()),
        'val_auc_mean': float(pd.to_numeric(stage_best['val_auc'], errors='coerce').mean()),
        'val_prauc_mean': float(pd.to_numeric(stage_best['val_prauc'], errors='coerce').mean()),
        'fairness_ratio_val_mean': float(pd.to_numeric(stage_rows.get('fairness_ratio_val', pd.Series(dtype=float)), errors='coerce').mean()) if not stage_rows.empty else float('nan'),
        'fairness_ratio_test_mean': float(pd.to_numeric(stage_rows.get('fairness_ratio_test', pd.Series(dtype=float)), errors='coerce').mean()) if not stage_rows.empty else float('nan'),
    })
stage_summary_df = pd.DataFrame(stage_summary_rows).sort_values('stage').reset_index(drop=True)
save_df_csv(stage_summary_df, PAPER_TABLES_DIR / 'paper_table_stage_summary.csv', 'Stage-wise tuning summary statistics')

use_test_metrics_for_paper = (
    LOG_TEST_DURING_TUNING
    and 'perf_recall_test' in merged_trial_summary_df.columns
    and merged_trial_summary_df['perf_recall_test'].notna().any()
)
paper_perf_col = 'perf_recall_test' if use_test_metrics_for_paper else 'recall_val'
paper_fair_col = 'fairness_ratio_test' if use_test_metrics_for_paper else 'fairness_ratio_val'

top_trials_export_df = merged_trial_summary_df.sort_values(paper_perf_col, ascending=False).head(max(10, TOP_K_TRIALS)).copy()
save_df_csv(top_trials_export_df, PAPER_TABLES_DIR / 'paper_table_top_trials.csv', 'Top trials by paper performance metric with fairness and hyperparameters')

# Final prediction exports (useful for later custom plots / appendix tables).
valid_predictions_df = build_prediction_export_df(valid_y_np, valid_probs, threshold_star, valid_sensitive_attributes_df, 'VALID')
test_predictions_df = build_prediction_export_df(test_y_np, test_probs, threshold_star, test_sensitive_attributes_df, 'TEST')
save_df_csv(valid_predictions_df, PAPER_PREDS_DIR / 'predictions_valid.csv', 'Per-row VALID predictions at selected threshold')
save_df_csv(test_predictions_df, PAPER_PREDS_DIR / 'predictions_test.csv', 'Per-row TEST predictions at selected threshold')

# Curves / threshold sweeps / calibration exports.
valid_roc_df = make_roc_df(valid_y_np, valid_probs)
test_roc_df = make_roc_df(test_y_np, test_probs)
valid_pr_df = make_pr_df(valid_y_np, valid_probs)
test_pr_df = make_pr_df(test_y_np, test_probs)
valid_threshold_sweep_df = make_threshold_sweep_df(valid_y_np, valid_probs, threshold_star)
test_threshold_sweep_df = make_threshold_sweep_df(test_y_np, test_probs, threshold_star)
valid_calibration_df = make_calibration_df(valid_y_np, valid_probs, n_bins=10)
test_calibration_df = make_calibration_df(test_y_np, test_probs, n_bins=10)

save_df_csv(valid_roc_df, PAPER_CURVES_DIR / 'curve_roc_valid.csv', 'ROC curve points (VALID)')
save_df_csv(test_roc_df, PAPER_CURVES_DIR / 'curve_roc_test.csv', 'ROC curve points (TEST)')
save_df_csv(valid_pr_df, PAPER_CURVES_DIR / 'curve_pr_valid.csv', 'PR curve points (VALID)')
save_df_csv(test_pr_df, PAPER_CURVES_DIR / 'curve_pr_test.csv', 'PR curve points (TEST)')
save_df_csv(valid_threshold_sweep_df, PAPER_CURVES_DIR / 'curve_threshold_sweep_valid.csv', 'Threshold sweep metrics (VALID)')
save_df_csv(test_threshold_sweep_df, PAPER_CURVES_DIR / 'curve_threshold_sweep_test.csv', 'Threshold sweep metrics (TEST)')
save_df_csv(valid_calibration_df, PAPER_CURVES_DIR / 'curve_calibration_valid.csv', 'Calibration bins (VALID)')
save_df_csv(test_calibration_df, PAPER_CURVES_DIR / 'curve_calibration_test.csv', 'Calibration bins (TEST)')

# Fairness exports: attribute summary + Aequitas disparity + subgroup threshold metrics.
fairness_attr_summary_df = build_attr_parity_summary_df(valid_fairness_attr_scores, test_fairness_attr_scores)
save_df_csv(fairness_attr_summary_df, PAPER_TABLES_DIR / 'paper_table_fairness_attr_parity_summary.csv', 'Aequitas FPR parity summary by attribute (VALID/TEST)')

valid_aequitas_export_df = prepare_aequitas_export_df(valid_aequitas_disparity_df, 'VALID')
test_aequitas_export_df = prepare_aequitas_export_df(test_aequitas_disparity_df, 'TEST')
if not valid_aequitas_export_df.empty:
    save_df_csv(valid_aequitas_export_df, PAPER_TABLES_DIR / 'paper_table_aequitas_disparity_valid.csv', 'Aequitas disparity table (VALID, tracked attributes)')
if not test_aequitas_export_df.empty:
    save_df_csv(test_aequitas_export_df, PAPER_TABLES_DIR / 'paper_table_aequitas_disparity_test.csv', 'Aequitas disparity table (TEST, tracked attributes)')

valid_subgroup_metrics_df = build_subgroup_metrics_df(valid_y_np, valid_probs, threshold_star, valid_sensitive_attributes_df, 'VALID')
test_subgroup_metrics_df = build_subgroup_metrics_df(test_y_np, test_probs, threshold_star, test_sensitive_attributes_df, 'TEST')
if not valid_subgroup_metrics_df.empty:
    save_df_csv(valid_subgroup_metrics_df, PAPER_TABLES_DIR / 'paper_table_subgroup_metrics_valid.csv', 'Subgroup metrics at selected threshold (VALID)')
if not test_subgroup_metrics_df.empty:
    save_df_csv(test_subgroup_metrics_df, PAPER_TABLES_DIR / 'paper_table_subgroup_metrics_test.csv', 'Subgroup metrics at selected threshold (TEST)')

print('Paper CSV/table exports complete.')
print('Tables dir :', PAPER_TABLES_DIR)
print('Curves dir :', PAPER_CURVES_DIR)
print('Preds dir  :', PAPER_PREDS_DIR)
display(final_metrics_df)
display(stage_summary_df)



## Block 37: Paper Diagrams - Core Curves and Operating Point

Generates paper-ready ROC/PR curves and threshold-sweep diagrams from the advanced experiment outputs.


In [ ]:
# ---------------- Paper plots: ROC/PR + threshold sweeps ----------------
if sns is not None:
    sns.set_theme(style='whitegrid', context='talk')

# ROC + PR panel
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=PLOT_DPI)
ax = axes[0]
ax.plot(valid_roc_df['fpr'], valid_roc_df['tpr'], label=f"VALID ROC (AUC={valid_auc:.3f})", linewidth=2)
ax.plot(test_roc_df['fpr'], test_roc_df['tpr'], label=f"TEST ROC (AUC={test_auc:.3f})", linewidth=2)
ax.axline((0, 0), slope=1, linestyle='--', color='gray', alpha=0.7)
ax.axvline(FPR_CAP, linestyle=':', color='red', alpha=0.8, label=f'FPR cap = {FPR_CAP:.2f}')
ax.scatter([valid_fpr], [valid_recall], s=60, color='tab:blue', edgecolor='black', zorder=5, label='VALID selected op point')
ax.scatter([fpr_test], [recall_test], s=60, color='tab:orange', edgecolor='black', zorder=5, label='TEST selected op point')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate / Recall')
ax.set_title('ROC Curves (VALID vs TEST)')
ax.legend(fontsize=10, loc='lower right')

ax = axes[1]
ax.plot(valid_pr_df['recall'], valid_pr_df['precision'], label=f"VALID PR (AP={valid_prauc:.3f})", linewidth=2)
ax.plot(test_pr_df['recall'], test_pr_df['precision'], label=f"TEST PR (AP={test_prauc:.3f})", linewidth=2)
ax.axhline(valid_y_np.mean(), linestyle='--', color='tab:blue', alpha=0.5, label=f'VALID prevalence={valid_y_np.mean():.3f}')
ax.axhline(test_y_np.mean(), linestyle='--', color='tab:orange', alpha=0.5, label=f'TEST prevalence={test_y_np.mean():.3f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (VALID vs TEST)')
ax.legend(fontsize=10, loc='best')

save_fig(fig, PAPER_FIGS_DIR / 'fig_roc_pr_curves_valid_test.png', 'ROC and PR curves (VALID vs TEST) with operating point markers')
print('Saved core curves panel.')

# Threshold sweep panel (VALID + TEST)
fig, axes = plt.subplots(2, 1, figsize=(14, 10), dpi=PLOT_DPI, sharex=True)
for ax, sweep_df, split_name in [
    (axes[0], valid_threshold_sweep_df, 'VALID'),
    (axes[1], test_threshold_sweep_df, 'TEST'),
]:
    ax.plot(sweep_df['threshold'], sweep_df['recall'], label='Recall', linewidth=2)
    ax.plot(sweep_df['threshold'], sweep_df['fpr'], label='FPR', linewidth=2)
    ax.plot(sweep_df['threshold'], sweep_df['precision'], label='Precision', linewidth=2, alpha=0.9)
    ax.axvline(float(threshold_star), linestyle='--', color='black', alpha=0.8, label=f'Selected thr={threshold_star:.3f}')
    ax.axhline(float(FPR_CAP), linestyle=':', color='red', alpha=0.8, label=f'FPR cap={FPR_CAP:.2f}')
    if 'is_best_recall_under_cap' in sweep_df.columns and sweep_df['is_best_recall_under_cap'].any():
        best_cap_row = sweep_df.loc[sweep_df['is_best_recall_under_cap'].astype(bool)].iloc[0]
        ax.scatter([best_cap_row['threshold']], [best_cap_row['recall']], s=60, edgecolor='black', zorder=5)
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel('Metric value')
    ax.set_title(f'Threshold sweep ({split_name})')
    ax.legend(fontsize=10, ncol=4, loc='upper center')
axes[1].set_xlabel('Decision threshold')
save_fig(fig, PAPER_FIGS_DIR / 'fig_threshold_sweeps_valid_test.png', 'Threshold sweep curves (Recall/FPR/Precision) for VALID and TEST')
print('Saved threshold sweep panel.')



## Block 38: Paper Diagrams - Score Distributions, Calibration, and Confusion Matrices

Generates paper-ready plots for score distributions, calibration behavior, and confusion matrices at the selected operating threshold.


In [ ]:
# ---------------- Paper plots: score distributions, calibration, confusion matrices ----------------
# Score distributions by class and split
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=PLOT_DPI, sharey=True)
for ax, pred_df, split_name in [
    (axes[0], valid_predictions_df, 'VALID'),
    (axes[1], test_predictions_df, 'TEST'),
]:
    for class_value, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
        mask = pred_df['y_true'] == class_value
        if mask.any():
            ax.hist(pred_df.loc[mask, 'y_prob'], bins=40, alpha=0.55, density=True, color=color, label=f'class={class_value}')
    ax.axvline(float(threshold_star), linestyle='--', color='black', alpha=0.8, label=f'thr={threshold_star:.3f}')
    ax.set_title(f'{split_name} score distribution by class')
    ax.set_xlabel('Predicted fraud probability')
    ax.set_ylabel('Density')
    ax.legend(fontsize=10)
save_fig(fig, PAPER_FIGS_DIR / 'fig_score_distributions_valid_test.png', 'Predicted score distributions by class for VALID and TEST')
print('Saved score distribution panel.')

# Calibration plot (reliability diagram)
fig, ax = plt.subplots(figsize=(8, 7), dpi=PLOT_DPI)
if not valid_calibration_df.empty:
    ax.plot(valid_calibration_df['mean_predicted_prob'], valid_calibration_df['fraction_positives'], marker='o', linewidth=2, label='VALID')
if not test_calibration_df.empty:
    ax.plot(test_calibration_df['mean_predicted_prob'], test_calibration_df['fraction_positives'], marker='o', linewidth=2, label='TEST')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.8, label='Perfect calibration')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed positive rate')
ax.set_title('Calibration (Reliability Diagram)')
ax.legend()
save_fig(fig, PAPER_FIGS_DIR / 'fig_calibration_valid_test.png', 'Calibration / reliability diagram for VALID and TEST')
print('Saved calibration plot.')

# Confusion matrix heatmaps (VALID + TEST)
def _plot_confusion_heatmap(ax, metrics_dict, title):
    cm = np.array([[metrics_dict['tn'], metrics_dict['fp']], [metrics_dict['fn'], metrics_dict['tp']]], dtype=float)
    if sns is not None:
        sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', cbar=False, ax=ax)
    else:
        im = ax.imshow(cm, cmap='Blues')
        for (r, c), value in np.ndenumerate(cm):
            ax.text(c, r, f'{int(value)}', ha='center', va='center')
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'], rotation=0)
    ax.set_title(title)
    ax.set_xlabel(f"FPR={metrics_dict['fpr']:.4f} | Recall={metrics_dict['recall']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=PLOT_DPI)
_plot_confusion_heatmap(axes[0], valid_metrics_at_thr, f'VALID confusion @ thr={threshold_star:.3f}')
_plot_confusion_heatmap(axes[1], test_metrics_at_thr, f'TEST confusion @ thr={threshold_star:.3f}')
save_fig(fig, PAPER_FIGS_DIR / 'fig_confusion_matrices_valid_test.png', 'Confusion matrices for VALID and TEST at selected threshold')
print('Saved confusion matrix panel.')



## Block 39: Paper Diagrams - Fairness and Subgroup Analysis

Generates fairness-focused plots (attribute parity bars, Aequitas disparity heatmaps, subgroup recall/FPR tables visualized) for the paper.


In [ ]:
# ---------------- Paper plots: fairness / subgroup analysis ----------------
# Attribute-level parity summary bar chart.
if not fairness_attr_summary_df.empty:
    parity_long_df = fairness_attr_summary_df.melt(
        id_vars=['attribute_name'],
        value_vars=[c for c in fairness_attr_summary_df.columns if c.endswith('_score')],
        var_name='metric', value_name='parity_score'
    )
    parity_long_df['split'] = parity_long_df['metric'].str.replace('_fpr_parity_score', '', regex=False).str.upper()

    fig, ax = plt.subplots(figsize=(10, 6), dpi=PLOT_DPI)
    if sns is not None:
        sns.barplot(data=parity_long_df, x='attribute_name', y='parity_score', hue='split', ax=ax)
    else:
        for idx, split_name in enumerate(sorted(parity_long_df['split'].unique())):
            rows = parity_long_df[parity_long_df['split'] == split_name]
            x = np.arange(len(rows)) + (idx - 0.5) * 0.2
            ax.bar(x, rows['parity_score'], width=0.35, label=split_name)
            ax.set_xticks(np.arange(len(rows)))
            ax.set_xticklabels(rows['attribute_name'], rotation=15)
    ax.axhline(1.0, linestyle='--', color='gray', alpha=0.8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Aequitas FPR parity score (ideal=1)')
    ax.set_xlabel('Sensitive attribute')
    ax.set_title('Fairness parity summary by attribute')
    ax.legend()
    save_fig(fig, PAPER_FIGS_DIR / 'fig_fairness_attr_parity_summary.png', 'Attribute-level Aequitas FPR parity summary for VALID and TEST')
    print('Saved fairness parity bar chart.')

# Aequitas disparity heatmaps (FPR disparity) by attribute/group.
def _make_disparity_pivot(disparity_df):
    if disparity_df is None or disparity_df.empty or 'attribute_name' not in disparity_df.columns or 'attribute_value' not in disparity_df.columns:
        return pd.DataFrame()
    work_df = disparity_df.copy()
    if 'fpr_disparity' not in work_df.columns:
        return pd.DataFrame()
    work_df['row_label'] = work_df['attribute_name'].astype(str) + '::' + work_df['attribute_value'].astype(str)
    out = work_df[['row_label', 'fpr_disparity']].copy()
    out['fpr_disparity'] = pd.to_numeric(out['fpr_disparity'], errors='coerce')
    return out.set_index('row_label')

valid_disp_pivot = _make_disparity_pivot(valid_aequitas_export_df)
test_disp_pivot = _make_disparity_pivot(test_aequitas_export_df)
if not valid_disp_pivot.empty or not test_disp_pivot.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.35 * max(len(valid_disp_pivot), len(test_disp_pivot), 1) + 2)), dpi=PLOT_DPI)
    for ax, pivot_df, title in [
        (axes[0], valid_disp_pivot, 'VALID Aequitas FPR disparity'),
        (axes[1], test_disp_pivot, 'TEST Aequitas FPR disparity'),
    ]:
        if pivot_df.empty:
            ax.axis('off')
            ax.set_title(title + ' (no data)')
            continue
        plot_df = pivot_df.copy()
        plot_df.columns = ['FPR disparity vs major group']
        if sns is not None:
            sns.heatmap(plot_df, annot=True, fmt='.2f', cmap='coolwarm', center=1.0, ax=ax)
        else:
            im = ax.imshow(plot_df.values, aspect='auto', cmap='coolwarm')
            for (r, c), value in np.ndenumerate(plot_df.values):
                ax.text(c, r, f'{value:.2f}', ha='center', va='center')
            ax.set_yticks(range(len(plot_df.index)))
            ax.set_yticklabels(plot_df.index)
            ax.set_xticks([0])
            ax.set_xticklabels(plot_df.columns, rotation=45, ha='right')
        ax.set_title(title)
    save_fig(fig, PAPER_FIGS_DIR / 'fig_aequitas_fpr_disparity_heatmaps.png', 'Aequitas FPR disparity heatmaps for VALID and TEST')
    print('Saved Aequitas disparity heatmaps.')

# Subgroup recall/FPR chart (focus on age group first, then all attrs combined as facet-like bars).
subgroup_plot_df = pd.concat([
    valid_subgroup_metrics_df if 'valid_subgroup_metrics_df' in globals() else pd.DataFrame(),
    test_subgroup_metrics_df if 'test_subgroup_metrics_df' in globals() else pd.DataFrame(),
], ignore_index=True)
if not subgroup_plot_df.empty:
    subgroup_plot_df = subgroup_plot_df.copy()
    subgroup_plot_df['label'] = subgroup_plot_df['attribute_name'].astype(str) + '::' + subgroup_plot_df['attribute_value'].astype(str)

    # Recall bars
    fig, ax = plt.subplots(figsize=(16, 8), dpi=PLOT_DPI)
    plot_order = subgroup_plot_df.sort_values(['attribute_name', 'attribute_value'])['label'].unique().tolist()
    if sns is not None:
        sns.barplot(data=subgroup_plot_df, x='label', y='recall', hue='split', order=plot_order, ax=ax)
    else:
        for split_name in ['VALID', 'TEST']:
            rows = subgroup_plot_df[subgroup_plot_df['split'] == split_name].set_index('label').reindex(plot_order).reset_index()
            x = np.arange(len(rows)) + (-0.18 if split_name == 'VALID' else 0.18)
            ax.bar(x, rows['recall'], width=0.35, label=split_name)
        ax.set_xticks(np.arange(len(plot_order)))
        ax.set_xticklabels(plot_order, rotation=70, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_title('Subgroup Recall at Selected Threshold')
    ax.set_xlabel('Subgroup')
    ax.set_ylabel('Recall')
    ax.tick_params(axis='x', rotation=70)
    ax.legend()
    save_fig(fig, PAPER_FIGS_DIR / 'fig_subgroup_recall_bars.png', 'Subgroup recall bars at selected threshold (VALID/TEST)')

    # FPR bars
    fig, ax = plt.subplots(figsize=(16, 8), dpi=PLOT_DPI)
    if sns is not None:
        sns.barplot(data=subgroup_plot_df, x='label', y='fpr', hue='split', order=plot_order, ax=ax)
    else:
        for split_name in ['VALID', 'TEST']:
            rows = subgroup_plot_df[subgroup_plot_df['split'] == split_name].set_index('label').reindex(plot_order).reset_index()
            x = np.arange(len(rows)) + (-0.18 if split_name == 'VALID' else 0.18)
            ax.bar(x, rows['fpr'], width=0.35, label=split_name)
        ax.set_xticks(np.arange(len(plot_order)))
        ax.set_xticklabels(plot_order, rotation=70, ha='right')
    ax.axhline(FPR_CAP, linestyle='--', color='red', alpha=0.8, label=f'FPR cap={FPR_CAP:.2f}')
    ax.set_ylim(0, 1.05)
    ax.set_title('Subgroup FPR at Selected Threshold')
    ax.set_xlabel('Subgroup')
    ax.set_ylabel('FPR')
    ax.tick_params(axis='x', rotation=70)
    ax.legend()
    save_fig(fig, PAPER_FIGS_DIR / 'fig_subgroup_fpr_bars.png', 'Subgroup FPR bars at selected threshold (VALID/TEST)')
    print('Saved subgroup recall/FPR bar charts.')




## Block 40: Paper Diagrams - Tuning Diagnostics

Generates tuning plots (distribution views, performance/fairness tradeoffs, cumulative best progression, and hyperparameter-metric correlations) for analysis and appendix figures.


In [ ]:
# ---------------- Paper plots: tuning / optimization diagnostics ----------------
# Rebuild merged tuning table if needed.
if 'merged_trial_summary_df' not in globals():
    merged_trial_summary_df = best_df.merge(fair_df, on=['stage', 'trial'], how='left', suffixes=('_best', '_fair'))

metric_perf_col = paper_perf_col if 'paper_perf_col' in globals() else ('perf_recall_test' if ('perf_recall_test' in merged_trial_summary_df.columns and merged_trial_summary_df['perf_recall_test'].notna().any()) else 'recall_val')
metric_fair_col = paper_fair_col if 'paper_fair_col' in globals() else ('fairness_ratio_test' if ('fairness_ratio_test' in merged_trial_summary_df.columns and merged_trial_summary_df['fairness_ratio_test'].notna().any()) else 'fairness_ratio_val')

# Stage distribution boxplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=PLOT_DPI)
if sns is not None:
    sns.boxplot(data=merged_trial_summary_df, x='stage', y=metric_perf_col, ax=axes[0])
    sns.boxplot(data=merged_trial_summary_df, x='stage', y=metric_fair_col, ax=axes[1])
else:
    merged_trial_summary_df.boxplot(column=metric_perf_col, by='stage', ax=axes[0])
    merged_trial_summary_df.boxplot(column=metric_fair_col, by='stage', ax=axes[1])
axes[0].set_title(f'Study label distribution: {metric_perf_col}')
axes[0].set_xlabel('Study label'); axes[0].set_ylabel(metric_perf_col)
axes[1].set_title(f'Study label distribution: {metric_fair_col}')
axes[1].set_xlabel('Study label'); axes[1].set_ylabel(metric_fair_col)
fig.suptitle('')
save_fig(fig, PAPER_FIGS_DIR / 'fig_stage_metric_distributions.png', 'Stage-wise metric distribution boxplots (performance and fairness)')
print('Saved stage metric distribution plot.')

# Pareto/tradeoff scatter with stage coloring and top-trial labels
fig, ax = plt.subplots(figsize=(10, 8), dpi=PLOT_DPI)
for stage_name, stage_df in merged_trial_summary_df.groupby('stage'):
    ax.scatter(stage_df[metric_perf_col], stage_df[metric_fair_col], alpha=0.7, label=stage_name)
annot_df = merged_trial_summary_df.sort_values(metric_perf_col, ascending=False).head(max(10, TOP_K_TRIALS)).copy()
for _, row in annot_df.iterrows():
    ax.annotate(f"{row['stage']}#{int(row['trial'])}", (row[metric_perf_col], row[metric_fair_col]), fontsize=8, alpha=0.8)
ax.set_xlabel(metric_perf_col)
ax.set_ylabel(metric_fair_col)
ax.set_title('Performance vs Fairness tradeoff across tuning trials')
ax.legend(fontsize=9)
save_fig(fig, PAPER_FIGS_DIR / 'fig_stage_tradeoff_scatter_annotated.png', 'Annotated performance-fairness tradeoff scatter across tuning stages')
print('Saved tradeoff scatter plot.')

# Cumulative best progression per stage (using best_df rows sorted by trial)
fig, ax = plt.subplots(figsize=(12, 6), dpi=PLOT_DPI)
for stage_name, stage_df in best_df.groupby('stage'):
    work = stage_df.copy().sort_values('trial')
    work['cummax_val_recall_at_fpr'] = pd.to_numeric(work['val_recall_at_fpr'], errors='coerce').cummax()
    ax.plot(work['trial'], work['cummax_val_recall_at_fpr'], marker='o', label=stage_name)
ax.set_xlabel('Trial number')
ax.set_ylabel('Cumulative best validation recall@FPR cap')
ax.set_title('Tuning progress (cumulative best)')
ax.legend()
save_fig(fig, PAPER_FIGS_DIR / 'fig_stage_cumulative_best_progress.png', 'Cumulative best recall progression by tuning stage')
print('Saved tuning progress plot.')

# Hyperparameter-metric correlation heatmap (best-per-trial rows only)
correlation_cols = [
    'd_token', 'n_blocks', 'n_heads', 'ffn_hidden', 'dropout',
    'lr', 'batch', 'val_auc', 'val_prauc', 'val_recall_at_fpr',
]
available_corr_cols = [c for c in correlation_cols if c in best_df.columns]
if len(available_corr_cols) >= 3:
    corr_df = best_df[available_corr_cols].apply(pd.to_numeric, errors='coerce')
    corr_matrix = corr_df.corr(numeric_only=True)
    save_df_csv(corr_matrix.reset_index().rename(columns={'index': 'variable'}), PAPER_TABLES_DIR / 'paper_table_hparam_metric_correlation.csv', 'Correlation matrix: hyperparameters vs validation metrics (best-per-trial rows)')

    fig, ax = plt.subplots(figsize=(14, 12), dpi=PLOT_DPI)
    if sns is not None:
        sns.heatmap(corr_matrix, cmap='coolwarm', center=0.0, ax=ax)
    else:
        im = ax.imshow(corr_matrix.values, cmap='coolwarm', aspect='auto')
        ax.set_xticks(range(len(corr_matrix.columns)))
        ax.set_xticklabels(corr_matrix.columns, rotation=90)
        ax.set_yticks(range(len(corr_matrix.index)))
        ax.set_yticklabels(corr_matrix.index)
    ax.set_title('Hyperparameter / metric correlation (best-per-trial rows)')
    save_fig(fig, PAPER_FIGS_DIR / 'fig_hparam_metric_correlation_heatmap.png', 'Correlation heatmap for hyperparameters and metrics')
    print('Saved hyperparameter correlation heatmap.')



## Block 41: Paper Artifact Manifest, Checklist, and Display Tables

Builds a manifest of all generated CSVs/figures, validates presence of core paper artifacts, and displays key tables for notebook review.


In [ ]:
# ---------------- Paper artifact manifest + checklist ----------------
paper_artifact_manifest_df = pd.DataFrame(paper_artifact_registry)
if paper_artifact_manifest_df.empty:
    print('No paper artifacts were registered yet.')
else:
    # Refresh existence flags in case files were overwritten.
    paper_artifact_manifest_df['exists'] = paper_artifact_manifest_df['path'].map(lambda p: int(Path(p).exists()))
    paper_artifact_manifest_df = paper_artifact_manifest_df.sort_values(['kind', 'path']).reset_index(drop=True)
    save_df_csv(paper_artifact_manifest_df, PAPER_ARTIFACT_DIR / 'paper_artifact_manifest.csv', 'Manifest of generated paper CSVs and figures')
    with open(PAPER_ARTIFACT_DIR / 'paper_artifact_manifest.json', 'w') as f:
        json.dump(paper_artifact_manifest_df.to_dict(orient='records'), f, indent=2)
    register_artifact(PAPER_ARTIFACT_DIR / 'paper_artifact_manifest.json', 'json', 'Manifest of generated paper artifacts (JSON)')

required_artifacts = [
    ('table_final_metrics', PAPER_TABLES_DIR / 'paper_table_final_metrics.csv'),
    ('table_split_summary', PAPER_TABLES_DIR / 'paper_table_dataset_split_summary.csv'),
    ('table_selected_params', PAPER_TABLES_DIR / 'paper_table_selected_params.csv'),
    ('table_stage_summary', PAPER_TABLES_DIR / 'paper_table_stage_summary.csv'),
    ('table_top_trials', PAPER_TABLES_DIR / 'paper_table_top_trials.csv'),
    ('table_fairness_attr', PAPER_TABLES_DIR / 'paper_table_fairness_attr_parity_summary.csv'),
    ('curve_roc_valid', PAPER_CURVES_DIR / 'curve_roc_valid.csv'),
    ('curve_roc_test', PAPER_CURVES_DIR / 'curve_roc_test.csv'),
    ('curve_pr_valid', PAPER_CURVES_DIR / 'curve_pr_valid.csv'),
    ('curve_pr_test', PAPER_CURVES_DIR / 'curve_pr_test.csv'),
    ('curve_threshold_valid', PAPER_CURVES_DIR / 'curve_threshold_sweep_valid.csv'),
    ('curve_threshold_test', PAPER_CURVES_DIR / 'curve_threshold_sweep_test.csv'),
    ('fig_roc_pr', PAPER_FIGS_DIR / 'fig_roc_pr_curves_valid_test.png'),
    ('fig_threshold_sweeps', PAPER_FIGS_DIR / 'fig_threshold_sweeps_valid_test.png'),
    ('fig_score_distributions', PAPER_FIGS_DIR / 'fig_score_distributions_valid_test.png'),
    ('fig_confusion_matrices', PAPER_FIGS_DIR / 'fig_confusion_matrices_valid_test.png'),
    ('fig_calibration', PAPER_FIGS_DIR / 'fig_calibration_valid_test.png'),
    ('fig_fairness_parity', PAPER_FIGS_DIR / 'fig_fairness_attr_parity_summary.png'),
    ('fig_tradeoff_scatter', PAPER_FIGS_DIR / 'fig_stage_tradeoff_scatter_annotated.png'),
    ('fig_tuning_progress', PAPER_FIGS_DIR / 'fig_stage_cumulative_best_progress.png'),
]

checklist_df = pd.DataFrame([
    {'artifact_key': key, 'path': str(path), 'exists': int(Path(path).exists())}
    for key, path in required_artifacts
])
save_df_csv(checklist_df, PAPER_ARTIFACT_DIR / 'paper_artifact_checklist.csv', 'Checklist of core paper artifacts and existence flags')

missing_artifacts_df = checklist_df[checklist_df['exists'] == 0].copy()
print(f"Paper artifact manifest entries: {len(paper_artifact_manifest_df)}")
print(f"Checklist complete: {(checklist_df['exists'] == 1).sum()}/{len(checklist_df)} present")
if not missing_artifacts_df.empty:
    print('Missing core artifacts:')
    display(missing_artifacts_df)
else:
    print('All core paper artifacts are present.')

print('Paper artifacts root:', PAPER_ARTIFACT_DIR)

# High-value notebook outputs for quick review.
display(final_metrics_df)
display(confusion_summary_df)
display(fairness_attr_summary_df)
if 'top_trials_export_df' in globals():
    display(top_trials_export_df.head(10))
if 'retrain_history_df' in globals() and isinstance(retrain_history_df, pd.DataFrame):
    display(retrain_history_df.tail(10))

